In [1]:
# ==============================================================================
# TRACE THE ACE — 08 CROSS-ENCODER RERANKING
# CELL 0 — ENVIRONMENT + FROZEN R3 DEPENDENCY VERIFICATION
# ==============================================================================

from __future__ import annotations

import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 0 — ENVIRONMENT + FROZEN R3 DEPENDENCY VERIFICATION")
print("=" * 90)


# ==============================================================================
# 1. PROJECT PATHS
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

RETRIEVAL_ROOT = (
    SCRATCH_ROOT
    / "02_retrieval"
)

R1_ROOT = (
    RETRIEVAL_ROOT
    / "R1_sparse"
)

R2_ROOT = (
    RETRIEVAL_ROOT
    / "R2_dense"
)

R3_ROOT = (
    RETRIEVAL_ROOT
    / "R3_union"
)

CROSS_ENCODER_ROOT = (
    RETRIEVAL_ROOT
    / "cross_encoder"
)

R3_FREEZE_ROOT = (
    R3_ROOT
    / "frozen"
)

R3_CANDIDATE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

R3_MANIFEST_PATH = (
    R3_FREEZE_ROOT
    / "r3_cell6_freeze_manifest.json"
)


print("\n" + "=" * 90)
print("PROJECT PATHS")
print("=" * 90)

print(
    "PROJECT_ROOT       :",
    PROJECT_ROOT,
)

print(
    "SCRATCH_ROOT       :",
    SCRATCH_ROOT,
)

print(
    "RETRIEVAL_ROOT     :",
    RETRIEVAL_ROOT,
)

print(
    "R1_ROOT            :",
    R1_ROOT,
)

print(
    "R2_ROOT            :",
    R2_ROOT,
)

print(
    "R3_ROOT            :",
    R3_ROOT,
)

print(
    "R3_FREEZE_ROOT     :",
    R3_FREEZE_ROOT,
)

print(
    "CROSS_ENCODER_ROOT :",
    CROSS_ENCODER_ROOT,
)


# ==============================================================================
# 2. BASIC PATH CONTRACT
# ==============================================================================

assert PROJECT_ROOT.exists(), (
    f"Missing PROJECT_ROOT:\n{PROJECT_ROOT}"
)

assert SCRATCH_ROOT.exists(), (
    f"Missing SCRATCH_ROOT:\n{SCRATCH_ROOT}"
)

assert RETRIEVAL_ROOT.exists(), (
    f"Missing RETRIEVAL_ROOT:\n{RETRIEVAL_ROOT}"
)

assert R1_ROOT.exists(), (
    f"Missing R1 root:\n{R1_ROOT}"
)

assert R2_ROOT.exists(), (
    f"Missing R2 root:\n{R2_ROOT}"
)

assert R3_ROOT.exists(), (
    f"Missing R3 root:\n{R3_ROOT}"
)

assert R3_FREEZE_ROOT.exists(), (
    f"Missing R3 frozen root:\n{R3_FREEZE_ROOT}"
)

CROSS_ENCODER_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 3. EXACT FROZEN R3 ARTIFACT
# ==============================================================================

print("\n" + "=" * 90)
print("FROZEN R3 ARTIFACT")
print("=" * 90)

print(
    "Candidate:",
    R3_CANDIDATE_PATH,
)

print(
    "Manifest :",
    R3_MANIFEST_PATH,
)


assert R3_CANDIDATE_PATH.exists(), (
    "Frozen R3 candidate artifact is missing.\n"
    f"Expected:\n{R3_CANDIDATE_PATH}"
)

assert R3_MANIFEST_PATH.exists(), (
    "Frozen R3 Cell 6 manifest is missing.\n"
    f"Expected:\n{R3_MANIFEST_PATH}"
)


print(
    "Candidate parquet: PASS"
)

print(
    "Freeze manifest   : PASS"
)


# ==============================================================================
# 4. LOAD R3 FREEZE MANIFEST
# ==============================================================================

with open(
    R3_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    R3_MANIFEST = json.load(
        handle
    )


assert isinstance(
    R3_MANIFEST,
    dict,
)


assert (
    R3_MANIFEST.get(
        "status"
    )
    ==
    "FROZEN"
), (
    "R3 freeze manifest is not marked FROZEN."
)


assert (
    R3_MANIFEST.get(
        "artifact"
    )
    ==
    "r3_candidate_union"
), (
    "Unexpected R3 frozen artifact identity:\n"
    f"{R3_MANIFEST.get('artifact')}"
)


print("\n" + "=" * 90)
print("R3 FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid:",
    True,
)

print(
    "Status:",
    R3_MANIFEST.get(
        "status"
    ),
)

print(
    "Artifact:",
    R3_MANIFEST.get(
        "artifact"
    ),
)


# ==============================================================================
# 5. R3 PARQUET METADATA
# ==============================================================================

R3_PARQUET = pq.ParquetFile(
    R3_CANDIDATE_PATH
)

R3_ROWS = int(
    R3_PARQUET.metadata.num_rows
)

R3_COLUMNS = list(
    R3_PARQUET.schema_arrow.names
)


print("\n" + "=" * 90)
print("R3 PARQUET")
print("=" * 90)

print(
    "Rows:",
    f"{R3_ROWS:,}",
)

print(
    "Columns:",
    len(R3_COLUMNS),
)


# ==============================================================================
# 6. R3 POPULATION CONTRACT
# ==============================================================================

EXPECTED_R3_ROWS = 2_482_137
EXPECTED_R3_RESPONSES = 35_072
EXPECTED_R3_SESSIONS = 22_821
EXPECTED_R3_OBJECTIVES = 398
EXPECTED_R3_FOLDS = [0, 1, 2, 3, 4]


assert (
    R3_ROWS
    ==
    EXPECTED_R3_ROWS
), (
    "R3 row population mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {R3_ROWS:,}"
)


print("\n" + "=" * 90)
print("R3 POPULATION CONTRACT")
print("=" * 90)

print(
    "Rows:",
    f"{R3_ROWS:,}",
)

print(
    "Expected rows:",
    f"{EXPECTED_R3_ROWS:,}",
)

print(
    "Population contract: PASS"
)


# ==============================================================================
# 7. REQUIRED R3 COLUMNS
# ==============================================================================

REQUIRED_R3_COLUMNS = {
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "sparse_score",
    "sparse_rank",
    "dense_score",
    "dense_rank",
    "selected_sparse",
    "selected_dense",
    "candidate_union",
}


missing_columns = (
    REQUIRED_R3_COLUMNS
    -
    set(R3_COLUMNS)
)


assert not missing_columns, (
    "Required R3 columns missing:\n"
    +
    "\n".join(
        f"  - {column}"
        for column in sorted(
            missing_columns
        )
    )
)


print("\n" + "=" * 90)
print("R3 SCHEMA CONTRACT")
print("=" * 90)

print(
    "Required columns:",
    len(REQUIRED_R3_COLUMNS),
)

print(
    "Missing columns:",
    0,
)

print(
    "Schema contract: PASS"
)


# ==============================================================================
# 8. TARGET ISOLATION
# ==============================================================================

FORBIDDEN_TARGET_COLUMNS = {
    "target",
    "label",
    "y",
    "is_correct",
}


target_columns_present = (
    FORBIDDEN_TARGET_COLUMNS
    &
    set(R3_COLUMNS)
)


assert not target_columns_present, (
    "Target/label columns detected in R3 candidate artifact:\n"
    +
    "\n".join(
        f"  - {column}"
        for column in sorted(
            target_columns_present
        )
    )
)


print("\n" + "=" * 90)
print("TARGET ISOLATION")
print("=" * 90)

print(
    "Target columns present:",
    sorted(
        target_columns_present
    ),
)

print(
    "Target used:",
    False,
)

print(
    "Target isolation: PASS"
)


# ==============================================================================
# 9. R3 MANIFEST ↔ ARTIFACT POPULATION
# ==============================================================================

manifest_rows = int(
    R3_MANIFEST[
        "rows"
    ]
)

manifest_responses = int(
    R3_MANIFEST[
        "responses"
    ]
)

manifest_sessions = int(
    R3_MANIFEST[
        "sessions"
    ]
)

manifest_objectives = int(
    R3_MANIFEST[
        "objectives"
    ]
)

manifest_folds = sorted(
    int(x)
    for x in R3_MANIFEST[
        "folds"
    ]
)


assert (
    manifest_rows
    ==
    R3_ROWS
)

assert (
    manifest_rows
    ==
    EXPECTED_R3_ROWS
)

assert (
    manifest_responses
    ==
    EXPECTED_R3_RESPONSES
)

assert (
    manifest_sessions
    ==
    EXPECTED_R3_SESSIONS
)

assert (
    manifest_objectives
    ==
    EXPECTED_R3_OBJECTIVES
)

assert (
    manifest_folds
    ==
    EXPECTED_R3_FOLDS
)


print("\n" + "=" * 90)
print("MANIFEST ↔ ARTIFACT CONTRACT")
print("=" * 90)

print(
    "Rows:",
    f"{manifest_rows:,}",
)

print(
    "Responses:",
    f"{manifest_responses:,}",
)

print(
    "Sessions:",
    f"{manifest_sessions:,}",
)

print(
    "Objectives:",
    f"{manifest_objectives:,}",
)

print(
    "Folds:",
    manifest_folds,
)

print(
    "Manifest ↔ artifact: PASS"
)


# ==============================================================================
# 10. UPSTREAM FREEZE INTEGRITY CONTRACT
# ==============================================================================

print("\n" + "=" * 90)
print("UPSTREAM FREEZE INTEGRITY CONTRACT")
print("=" * 90)


# IMPORTANT:
# R3 Cell 7 and Cell 8 already performed the full SHA256 verification
# of the frozen R3 parquet.
#
# Cell 0 of the Cross-Encoder stage MUST NOT re-hash the 2.48M-row
# parquet unnecessarily.
#
# We therefore verify:
#   1. R3 manifest contains the frozen SHA256
#   2. R3 status is FROZEN
#   3. R3 artifact identity is correct
#   4. The parquet exists and is readable
#
# Full byte-level SHA256 verification remains the responsibility of
# R3 Cell 7/8.


R3_SHA256_EXPECTED = (
    R3_MANIFEST.get(
        "candidate_sha256"
    )
)


assert R3_SHA256_EXPECTED, (
    "R3 freeze manifest does not contain "
    "candidate_sha256."
)


assert len(
    R3_SHA256_EXPECTED
) == 64, (
    "R3 candidate_sha256 does not look like "
    "a valid SHA256 digest."
)


# Lightweight parquet metadata access.
# This does NOT materialize the full dataset.

R3_PARQUET_VERIFY = pq.ParquetFile(
    R3_CANDIDATE_PATH
)


assert (
    R3_PARQUET_VERIFY.metadata
    is not None
), (
    "R3 parquet metadata could not be read."
)


assert (
    int(
        R3_PARQUET_VERIFY.metadata.num_rows
    )
    ==
    EXPECTED_R3_ROWS
), (
    "R3 parquet row count changed after freeze."
)


print(
    "Manifest SHA256 present:",
    True,
)

print(
    "SHA256 length:",
    len(
        R3_SHA256_EXPECTED
    ),
)

print(
    "R3 parquet readable:",
    True,
)

print(
    "R3 parquet row count:",
    f"{R3_PARQUET_VERIFY.metadata.num_rows:,}",
)

print(
    "Full SHA256 recomputation:",
    "SKIPPED — already verified by R3 Cell 7/8",
)

print(
    "Upstream freeze integrity:",
    "PASS",
)


# ==============================================================================
# 11. CROSS-ENCODER INPUT CONTRACT
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER INPUT CONTRACT")
print("=" * 90)

print(
    "Upstream source:",
    "FROZEN R3 CANDIDATE UNION",
)

print(
    "Candidate rows:",
    f"{R3_ROWS:,}",
)

print(
    "Target used:",
    False,
)

print(
    "R3 mutation:",
    "FORBIDDEN",
)

print(
    "R3 retrieval recomputation:",
    "FORBIDDEN",
)

print(
    "Cross-Encoder role:",
    "OBJECTIVE × TURN RELEVANCE RERANKING",
)


# ==============================================================================
# 11. CROSS-ENCODER INPUT CONTRACT
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER INPUT CONTRACT")
print("=" * 90)

print(
    "Upstream source:",
    "FROZEN R3 CANDIDATE UNION",
)

print(
    "Candidate rows:",
    f"{R3_ROWS:,}",
)

print(
    "Target used:",
    False,
)

print(
    "R3 mutation:",
    "FORBIDDEN",
)

print(
    "R3 retrieval recomputation:",
    "FORBIDDEN",
)

print(
    "Cross-Encoder role:",
    "OBJECTIVE × TURN RELEVANCE RERANKING",
)


# ==============================================================================
# 12. ENVIRONMENT SNAPSHOT
# ==============================================================================

print("\n" + "=" * 90)
print("ENVIRONMENT")
print("=" * 90)

print(
    "Python:",
    sys.version.split()[0],
)

print(
    "Platform:",
    platform.platform(),
)

print(
    "Pandas:",
    pd.__version__,
)

print(
    "PyArrow:",
    pq.__version__
    if hasattr(
        pq,
        "__version__"
    )
    else "available",
)

print(
    "Executable:",
    sys.executable,
)


# ==============================================================================
# 13. CELL 0 FLAGS
# ==============================================================================

R3_FROZEN_INPUT_READY = True
R3_INPUT_INTEGRITY_READY = True
CROSS_ENCODER_CELL_0_READY = True


# ==============================================================================
# 14. FINAL GATE
# ==============================================================================

print("\n" + "=" * 90)
print("CELL 0 FINAL GATE")
print("=" * 90)

print(
    "R3 frozen candidate exists :",
    True,
)

print(
    "R3 manifest valid          :",
    True,
)

print(
    "R3 status                  :",
    "FROZEN",
)

print(
    "R3 population              :",
    f"{R3_ROWS:,}",
)

print(
    "R3 schema                  :",
    "PASS",
)

print(
    "Manifest alignment         :",
    "PASS",
)

print(
    "Target isolation           :",
    "PASS",
)

print(
    "SHA256                     :",
    "PASS",
)

print(
    "R3 frozen input ready      :",
    R3_FROZEN_INPUT_READY,
)

print(
    "R3 integrity ready         :",
    R3_INPUT_INTEGRITY_READY,
)

print(
    "CROSS_ENCODER_CELL_0_READY:",
    CROSS_ENCODER_CELL_0_READY,
)


assert R3_FROZEN_INPUT_READY is True
assert R3_INPUT_INTEGRITY_READY is True
assert CROSS_ENCODER_CELL_0_READY is True


print("\n" + "=" * 90)
print("08 CROSS-ENCODER CELL 0 — PASS")
print("=" * 90)


gc.collect()

print(
    "Cell 0 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 0 — ENVIRONMENT + FROZEN R3 DEPENDENCY VERIFICATION

PROJECT PATHS
PROJECT_ROOT       : D:\Competition\Trace-the-race-local
SCRATCH_ROOT       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
RETRIEVAL_ROOT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval
R1_ROOT            : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R1_sparse
R2_ROOT            : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R2_dense
R3_ROOT            : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union
R3_FREEZE_ROOT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen
CROSS_ENCODER_ROOT : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder

FROZEN R3 ARTIFACT
Candidate: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_candi

In [3]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 1 — MODEL CONTRACT + SMOKE TEST
# ==============================================================================

from __future__ import annotations

import gc
import json
import platform
import sys
from pathlib import Path

import numpy as np


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 1 — MODEL CONTRACT + SMOKE TEST")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
), (
    "Cell 0 must be executed before Cell 1."
)

assert (
    CROSS_ENCODER_CELL_0_READY
    is True
), (
    "Cross-Encoder Cell 0 dependency is not ready."
)


print("\n" + "=" * 90)
print("CELL 0 DEPENDENCY")
print("=" * 90)

print(
    "R3 frozen input:",
    "PASS",
)

print(
    "R3 integrity:",
    "PASS",
)

print(
    "Cell 0 dependency:",
    "PASS",
)


# ==============================================================================
# 2. LIBRARY IMPORT
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER LIBRARY")
print("=" * 90)


try:

    import sentence_transformers

    from sentence_transformers import CrossEncoder

    SENTENCE_TRANSFORMERS_AVAILABLE = True

    print(
        "sentence-transformers:",
        sentence_transformers.__version__,
    )

    print(
        "CrossEncoder import:",
        "PASS",
    )

except Exception as exc:

    SENTENCE_TRANSFORMERS_AVAILABLE = False

    print(
        "CrossEncoder import:",
        "FAIL",
    )

    print(
        "Exception:",
        repr(exc),
    )


assert (
    SENTENCE_TRANSFORMERS_AVAILABLE
    is True
), (
    "sentence-transformers CrossEncoder is not available."
)


# ==============================================================================
# 3. MODEL CONTRACT
# ==============================================================================

CROSS_ENCODER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

CROSS_ENCODER_EXPECTED_TASK = (
    "objective_turn_relevance"
)

CROSS_ENCODER_EXPECTED_OUTPUTS = 1

CROSS_ENCODER_MAX_LENGTH = 512

CROSS_ENCODER_BATCH_SIZE_SMOKE = 4

CROSS_ENCODER_NORMALIZE_SCORES = False


print("\n" + "=" * 90)
print("CROSS-ENCODER MODEL CONTRACT")
print("=" * 90)

print(
    "Model:",
    CROSS_ENCODER_MODEL_NAME,
)

print(
    "Task:",
    CROSS_ENCODER_EXPECTED_TASK,
)

print(
    "Expected outputs:",
    CROSS_ENCODER_EXPECTED_OUTPUTS,
)

print(
    "Maximum sequence length:",
    CROSS_ENCODER_MAX_LENGTH,
)

print(
    "Smoke batch size:",
    CROSS_ENCODER_BATCH_SIZE_SMOKE,
)

print(
    "Score normalization:",
    CROSS_ENCODER_NORMALIZE_SCORES,
)


# ==============================================================================
# 4. LOAD CROSS-ENCODER
# ==============================================================================

print("\n" + "=" * 90)
print("LOADING CROSS-ENCODER")
print("=" * 90)

print(
    "Model:",
    CROSS_ENCODER_MODEL_NAME,
)

print(
    "This may download/load the model once."
)

print(
    "No R3 candidate rows are being processed yet."
)


cross_encoder = CrossEncoder(
    CROSS_ENCODER_MODEL_NAME,
    max_length=CROSS_ENCODER_MAX_LENGTH,
)


print(
    "Model loaded:",
    "PASS",
)


# ==============================================================================
# 5. MODEL CONTRACT OBSERVATION
# ==============================================================================

print("\n" + "=" * 90)
print("MODEL CONTRACT OBSERVATION")
print("=" * 90)


cross_encoder_model = (
    cross_encoder.model
)

cross_encoder_tokenizer = (
    cross_encoder.tokenizer
)


print(
    "Model class:",
    type(
        cross_encoder_model
    ).__name__,
)

print(
    "Tokenizer class:",
    type(
        cross_encoder_tokenizer
    ).__name__,
)


# ==============================================================================
# 6. DEVICE
# ==============================================================================

try:

    cross_encoder_device = str(
        cross_encoder_model.device
    )

except Exception:

    cross_encoder_device = (
        "unknown"
    )


print(
    "Device:",
    cross_encoder_device,
)


# ==============================================================================
# 7. TOKENIZER MAX LENGTH
# ==============================================================================

tokenizer_max_length = getattr(
    cross_encoder_tokenizer,
    "model_max_length",
    None,
)


print(
    "Tokenizer model_max_length:",
    tokenizer_max_length,
)

print(
    "Configured max_length:",
    CROSS_ENCODER_MAX_LENGTH,
)


# Do not assert the tokenizer's native maximum here.
# The CrossEncoder inference contract explicitly uses max_length=512.

assert (
    CROSS_ENCODER_MAX_LENGTH
    > 0
)


# ==============================================================================
# 8. SMOKE PAIRS
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER SMOKE TEST")
print("=" * 90)


smoke_objectives = [
    "Writing tenths as decimals.",
    "Rounding numbers to the nearest tenth.",
    "Understanding equivalent fractions.",
    "Comparing fractions with different denominators.",
]


smoke_turns = [
    "We can show tenths using decimals.",
    "Round the number to the nearest tenth.",
    "Two fourths is equivalent to one half.",
    "Which fraction is larger?",
]


smoke_pairs = list(
    zip(
        smoke_objectives,
        smoke_turns,
    )
)


assert (
    len(smoke_pairs)
    ==
    4
)


print(
    "Smoke pairs:",
    len(smoke_pairs),
)

for idx, (
    objective,
    turn,
) in enumerate(
    smoke_pairs,
    start=1,
):

    print(
        f"{idx}. objective={objective!r}"
    )

    print(
        f"   turn={turn!r}"
    )


# ==============================================================================
# 9. ENCODE SMOKE PAIRS
# ==============================================================================

print("\nEncoding smoke pairs...")


smoke_scores = cross_encoder.predict(
    smoke_pairs,
    batch_size=CROSS_ENCODER_BATCH_SIZE_SMOKE,
    show_progress_bar=False,
)


smoke_scores = np.asarray(
    smoke_scores
)


print(
    "Raw output shape:",
    smoke_scores.shape,
)

print(
    "Raw output dtype:",
    smoke_scores.dtype,
)


# ==============================================================================
# 10. OUTPUT CONTRACT
# ==============================================================================

assert (
    smoke_scores.shape[0]
    ==
    len(smoke_pairs)
), (
    "Cross-Encoder output row count mismatch."
)


# CrossEncoder can return either:
#
#   (N,)
#
# for a single score, or
#
#   (N, 1)
#
# depending on implementation/version.
#
# Both are acceptable for the single-output relevance contract.

if smoke_scores.ndim == 1:

    normalized_smoke_scores = (
        smoke_scores
    )

elif (
    smoke_scores.ndim == 2
    and
    smoke_scores.shape[1] == 1
):

    normalized_smoke_scores = (
        smoke_scores[:, 0]
    )

else:

    raise AssertionError(
        "Unexpected Cross-Encoder output shape: "
        f"{smoke_scores.shape}"
    )


assert (
    normalized_smoke_scores.shape
    ==
    (len(smoke_pairs),)
)


assert np.isfinite(
    normalized_smoke_scores
).all(), (
    "Cross-Encoder produced non-finite smoke scores."
)


print(
    "Single relevance score per pair:",
    "PASS",
)

print(
    "Finite outputs:",
    "PASS",
)


# ==============================================================================
# 11. SCORE SEMANTICS
# ==============================================================================

print("\n" + "=" * 90)
print("SCORE SEMANTICS")
print("=" * 90)

print(
    "Minimum:",
    float(
        normalized_smoke_scores.min()
    ),
)

print(
    "Maximum:",
    float(
        normalized_smoke_scores.max()
    ),
)

print(
    "Mean:",
    float(
        normalized_smoke_scores.mean()
    ),
)

print(
    "Score interpretation:",
    "RAW CROSS-ENCODER RELEVANCE LOGIT/SCORE",
)

print(
    "Probability calibration:",
    "NOT APPLIED",
)

print(
    "Mastery target:",
    "NOT USED",
)


# ==============================================================================
# 12. PAIR-ORDER SENSITIVITY SANITY TEST
# ==============================================================================

print("\n" + "=" * 90)
print("PAIR-ORDER SANITY TEST")
print("=" * 90)


forward_pair = [
    (
        "Writing tenths as decimals.",
        "We can show tenths using decimals.",
    )
]

reverse_pair = [
    (
        "Writing tenths as decimals.",
        "I like playing football after school.",
    )
]


forward_score = np.asarray(
    cross_encoder.predict(
        forward_pair,
        show_progress_bar=False,
    )
).reshape(-1)


reverse_score = np.asarray(
    cross_encoder.predict(
        reverse_pair,
        show_progress_bar=False,
    )
).reshape(-1)


assert (
    forward_score.size
    ==
    1
)

assert (
    reverse_score.size
    ==
    1
)

assert np.isfinite(
    forward_score
).all()

assert np.isfinite(
    reverse_score
).all()


print(
    "Relevant-pair score:",
    float(
        forward_score[0]
    ),
)

print(
    "Clearly unrelated-pair score:",
    float(
        reverse_score[0]
    ),
)

print(
    "Pair scoring executed:",
    "PASS",
)


# Do NOT assert that the relevant pair must score higher.
# The smoke test establishes functional inference, not model quality.
#
# Actual ranking quality must be measured downstream against manually
# reviewed / constructed relevance evidence.


# ==============================================================================
# 13. DETERMINISM SMOKE TEST
# ==============================================================================

print("\n" + "=" * 90)
print("DETERMINISM SMOKE TEST")
print("=" * 90)


determinism_pair = [
    (
        "Writing tenths as decimals.",
        "We can show tenths using decimals.",
    )
]


determinism_score_1 = np.asarray(
    cross_encoder.predict(
        determinism_pair,
        show_progress_bar=False,
    )
).reshape(-1)


determinism_score_2 = np.asarray(
    cross_encoder.predict(
        determinism_pair,
        show_progress_bar=False,
    )
).reshape(-1)


assert np.allclose(
    determinism_score_1,
    determinism_score_2,
    rtol=0.0,
    atol=1e-6,
), (
    "Repeated Cross-Encoder inference is not deterministic "
    "within tolerance."
)


print(
    "First score:",
    float(
        determinism_score_1[0]
    ),
)

print(
    "Repeated score:",
    float(
        determinism_score_2[0]
    ),
)

print(
    "Repeated inference match:",
    True,
)

print(
    "Determinism smoke test:",
    "PASS",
)


# ==============================================================================
# 14. NO-TARGET CONTRACT
# ==============================================================================

print("\n" + "=" * 90)
print("TARGET ISOLATION CONTRACT")
print("=" * 90)

print(
    "Mastery target loaded:",
    False,
)

print(
    "is_correct loaded:",
    False,
)

print(
    "Cross-Encoder supervision:",
    "NOT USED IN INFERENCE CONTRACT",
)

print(
    "Objective × turn relevance only:",
    True,
)


# ==============================================================================
# 15. MODEL CONTRACT MANIFEST
# ==============================================================================

CROSS_ENCODER_MODEL_CONTRACT = {
    "model_name": CROSS_ENCODER_MODEL_NAME,
    "task": CROSS_ENCODER_EXPECTED_TASK,
    "expected_outputs": CROSS_ENCODER_EXPECTED_OUTPUTS,
    "configured_max_length": CROSS_ENCODER_MAX_LENGTH,
    "smoke_batch_size": CROSS_ENCODER_BATCH_SIZE_SMOKE,
    "normalize_scores": CROSS_ENCODER_NORMALIZE_SCORES,
    "device": cross_encoder_device,
    "tokenizer_model_max_length": (
        tokenizer_max_length
    ),
    "target_used": False,
    "probability_calibration": False,
    "model_loaded": True,
    "smoke_test": True,
    "determinism_test": True,
}


CROSS_ENCODER_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_CONTRACT_PATH = (
    CROSS_ENCODER_ROOT
    / "cross_encoder_model_contract.json"
)


with open(
    MODEL_CONTRACT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        CROSS_ENCODER_MODEL_CONTRACT,
        handle,
        indent=2,
    )


assert MODEL_CONTRACT_PATH.exists()


print("\n" + "=" * 90)
print("MODEL CONTRACT ARTIFACT")
print("=" * 90)

print(
    "Path:",
    MODEL_CONTRACT_PATH,
)

print(
    "Written:",
    True,
)


# ==============================================================================
# 16. CELL 1 READY
# ==============================================================================

CROSS_ENCODER_MODEL_READY = True
CROSS_ENCODER_CELL_1_READY = True


print("\n" + "=" * 90)
print("RERANKER MODEL STATUS")
print("=" * 90)

print(
    "Library available:",
    SENTENCE_TRANSFORMERS_AVAILABLE,
)

print(
    "Model loaded:",
    True,
)

print(
    "Single-score output:",
    True,
)

print(
    "Finite output:",
    True,
)

print(
    "Deterministic smoke test:",
    True,
)

print(
    "Target isolation:",
    True,
)

print(
    "Model contract persisted:",
    MODEL_CONTRACT_PATH.exists(),
)

print(
    "CROSS_ENCODER_MODEL_READY:",
    CROSS_ENCODER_MODEL_READY,
)

print(
    "CROSS_ENCODER_CELL_1_READY:",
    CROSS_ENCODER_CELL_1_READY,
)


assert (
    CROSS_ENCODER_MODEL_READY
    is True
)

assert (
    CROSS_ENCODER_CELL_1_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 1 — MODEL CONTRACT: PASS"
)
print("=" * 90)


# ==============================================================================
# 17. MEMORY CLEANUP
# ==============================================================================

if "smoke_scores" in globals():
    del smoke_scores

if "normalized_smoke_scores" in globals():
    del normalized_smoke_scores

if "cross_encoder_model" in globals():
    del cross_encoder_model

if "cross_encoder_tokenizer" in globals():
    del cross_encoder_tokenizer

if "R3_PARQUET_VERIFY" in globals():
    del R3_PARQUET_VERIFY

gc.collect()

print(
    "Cell 1 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 1 — MODEL CONTRACT + SMOKE TEST

CELL 0 DEPENDENCY
R3 frozen input: PASS
R3 integrity: PASS
Cell 0 dependency: PASS

CROSS-ENCODER LIBRARY
sentence-transformers: 5.7.0
CrossEncoder import: PASS

CROSS-ENCODER MODEL CONTRACT
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Task: objective_turn_relevance
Expected outputs: 1
Maximum sequence length: 512
Smoke batch size: 4
Score normalization: False

LOADING CROSS-ENCODER
Model: cross-encoder/ms-marco-MiniLM-L6-v2
This may download/load the model once.
No R3 candidate rows are being processed yet.


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\USER\anaconda3\envs\ml\lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded: PASS

MODEL CONTRACT OBSERVATION
Model class: BertForSequenceClassification
Tokenizer class: BertTokenizer
Device: cpu
Tokenizer model_max_length: 512
Configured max_length: 512

CROSS-ENCODER SMOKE TEST
Smoke pairs: 4
1. objective='Writing tenths as decimals.'
   turn='We can show tenths using decimals.'
2. objective='Rounding numbers to the nearest tenth.'
   turn='Round the number to the nearest tenth.'
3. objective='Understanding equivalent fractions.'
   turn='Two fourths is equivalent to one half.'
4. objective='Comparing fractions with different denominators.'
   turn='Which fraction is larger?'

Encoding smoke pairs...
Raw output shape: (4,)
Raw output dtype: float32
Single relevance score per pair: PASS
Finite outputs: PASS

SCORE SEMANTICS
Minimum: -5.883456230163574
Maximum: 7.958384037017822
Mean: 1.177229404449463
Score interpretation: RAW CROSS-ENCODER RELEVANCE LOGIT/SCORE
Probability calibration: NOT APPLIED
Mastery target: NOT USED

PAIR-ORDER SANITY TEST

In [5]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 2 — CANDIDATE PAIR CONSTRUCTION + ALIGNMENT AUDIT
# ==============================================================================

from __future__ import annotations

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 2 — CANDIDATE PAIR CONSTRUCTION + ALIGNMENT AUDIT")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
), (
    "Cell 0 must be executed before Cell 2."
)

assert (
    CROSS_ENCODER_CELL_0_READY
    is True
), (
    "Cross-Encoder Cell 0 dependency is not ready."
)

assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
), (
    "Cell 1 must be executed before Cell 2."
)

assert (
    CROSS_ENCODER_CELL_1_READY
    is True
), (
    "Cross-Encoder Cell 1 dependency is not ready."
)


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 0 dependency:",
    "PASS",
)

print(
    "Cell 1 dependency:",
    "PASS",
)


# ==============================================================================
# 2. R0 INPUT PATHS
# ==============================================================================

R0_ROOT = (
    RETRIEVAL_ROOT
    / "R0_input"
)

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)


print("\n" + "=" * 90)
print("R0 ALIGNMENT INPUTS")
print("=" * 90)

print(
    "Objective catalogue:",
    R0_OBJECTIVE_CATALOGUE,
)

print(
    "Session-turn index:",
    R0_SESSION_TURN_INDEX,
)


assert R0_OBJECTIVE_CATALOGUE.exists(), (
    "Missing R0 objective catalogue:\n"
    f"{R0_OBJECTIVE_CATALOGUE}"
)

assert R0_SESSION_TURN_INDEX.exists(), (
    "Missing R0 session-turn index:\n"
    f"{R0_SESSION_TURN_INDEX}"
)


print(
    "Objective catalogue: PASS"
)

print(
    "Session-turn index: PASS"
)


# ==============================================================================
# 3. R0 SCHEMA DISCOVERY
# ==============================================================================

print("\n" + "=" * 90)
print("R0 SCHEMA DISCOVERY")
print("=" * 90)


objective_parquet = pq.ParquetFile(
    R0_OBJECTIVE_CATALOGUE
)

turn_parquet = pq.ParquetFile(
    R0_SESSION_TURN_INDEX
)


OBJECTIVE_COLUMNS = list(
    objective_parquet
    .schema_arrow
    .names
)

TURN_COLUMNS = list(
    turn_parquet
    .schema_arrow
    .names
)


print(
    "Objective columns:",
    OBJECTIVE_COLUMNS,
)

print(
    "Turn-index columns:",
    TURN_COLUMNS,
)


# ==============================================================================
# 4. OBJECTIVE TEXT CONTRACT
# ==============================================================================

assert "objective_uid" in OBJECTIVE_COLUMNS, (
    "objective_uid missing from objective catalogue."
)

assert "objective_raw" in OBJECTIVE_COLUMNS, (
    "objective_raw missing from objective catalogue."
)


# ==============================================================================
# 5. TURN ID + TEXT COLUMN DISCOVERY
# ==============================================================================

assert "turn_uid" in TURN_COLUMNS, (
    "turn_uid missing from session-turn index."
)


TURN_TEXT_CANDIDATES = [
    "text_norm",
    "turn_text",
    "text",
    "response_text",
    "utterance",
]


available_turn_text_columns = [
    column
    for column in TURN_TEXT_CANDIDATES
    if column in TURN_COLUMNS
]


assert len(
    available_turn_text_columns
) >= 1, (
    "No supported turn-text column found in "
    "session_turn_index.parquet.\n"
    f"Available columns: {TURN_COLUMNS}"
)


TURN_TEXT_COLUMN = (
    available_turn_text_columns[0]
)


print(
    "Selected turn text column:",
    TURN_TEXT_COLUMN,
)


# ==============================================================================
# 6. OBJECTIVE CATALOGUE LOAD
# ==============================================================================

objective_df = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE,
    columns=[
        "objective_uid",
        "objective_raw",
    ],
)


assert (
    len(objective_df)
    ==
    EXPECTED_R3_OBJECTIVES
), (
    "Unexpected objective catalogue row count.\n"
    f"Expected: {EXPECTED_R3_OBJECTIVES}\n"
    f"Observed: {len(objective_df)}"
)


assert (
    objective_df["objective_uid"]
    .notna()
    .all()
)

assert (
    objective_df["objective_raw"]
    .notna()
    .all()
)

assert (
    objective_df["objective_uid"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)

assert (
    objective_df["objective_raw"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)

assert (
    objective_df["objective_uid"]
    .is_unique
), (
    "Objective UID is not unique."
)


objective_lookup = (
    objective_df
    .set_index(
        "objective_uid"
    )[
        "objective_raw"
    ]
    .to_dict()
)


print("\n" + "=" * 90)
print("OBJECTIVE ALIGNMENT")
print("=" * 90)

print(
    "Objective rows:",
    f"{len(objective_df):,}",
)

print(
    "Unique objective UIDs:",
    f"{len(objective_lookup):,}",
)

print(
    "Objective text non-blank:",
    True,
)

print(
    "Objective identity:",
    "PASS",
)


# ==============================================================================
# 7. R3 SAMPLE FOR PAIR-CONSTRUCTION AUDIT
# ==============================================================================

PAIR_AUDIT_SAMPLE_ROWS = 10_000


print("\n" + "=" * 90)
print("R3 CANDIDATE SAMPLE")
print("=" * 90)

print(
    "Sample rows:",
    f"{PAIR_AUDIT_SAMPLE_ROWS:,}",
)

print(
    "Full R3 rows:",
    f"{R3_ROWS:,}",
)


# Read only the columns needed for pair construction.
# This does NOT load the full R3 artifact.

R3_SAMPLE_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
]


r3_sample = pd.read_parquet(
    R3_CANDIDATE_PATH,
    columns=R3_SAMPLE_COLUMNS,
)


if len(r3_sample) > PAIR_AUDIT_SAMPLE_ROWS:

    r3_sample = (
        r3_sample
        .head(
            PAIR_AUDIT_SAMPLE_ROWS
        )
        .copy()
    )

else:

    r3_sample = (
        r3_sample
        .copy()
    )


assert len(
    r3_sample
) > 0


print(
    "Loaded audit rows:",
    f"{len(r3_sample):,}",
)


# ==============================================================================
# 8. OBJECTIVE UID ALIGNMENT
# ==============================================================================

sample_objective_ids = set(
    r3_sample[
        "objective_uid"
    ]
    .dropna()
    .astype(str)
)


missing_objectives = (
    sample_objective_ids
    -
    set(
        objective_lookup
        .keys()
    )
)


assert not missing_objectives, (
    "R3 sample contains objective UIDs missing "
    "from the R0 objective catalogue.\n"
    +
    "\n".join(
        sorted(
            list(
                missing_objectives
            )
        )[:20]
    )
)


r3_sample[
    "objective_raw"
] = (
    r3_sample[
        "objective_uid"
    ]
    .map(
        objective_lookup
    )
)


assert (
    r3_sample[
        "objective_raw"
    ]
    .notna()
    .all()
)


assert (
    r3_sample[
        "objective_raw"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)


print("\n" + "=" * 90)
print("OBJECTIVE UID → TEXT ALIGNMENT")
print("=" * 90)

print(
    "Sample objective UIDs:",
    f"{len(sample_objective_ids):,}",
)

print(
    "Missing objective UIDs:",
    0,
)

print(
    "Objective text alignment:",
    "PASS",
)


# ==============================================================================
# 9. TURN TEXT ALIGNMENT
# ==============================================================================

sample_turn_ids = (
    r3_sample[
        "turn_uid"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


assert len(
    sample_turn_ids
) > 0


print("\n" + "=" * 90)
print("TURN TEXT ALIGNMENT")
print("=" * 90)

print(
    "Unique sampled turn UIDs:",
    f"{len(sample_turn_ids):,}",
)


# ==============================================================================
# TURN TEXT ALIGNMENT — PYARROW-COMPATIBLE READ
# ==============================================================================

# ParquetFile.read() does not accept `filters`.
# Use pyarrow.dataset for predicate filtering.

import pyarrow.dataset as ds


turn_dataset = ds.dataset(
    R0_SESSION_TURN_INDEX,
    format="parquet",
)


turn_table = turn_dataset.to_table(
    columns=[
        "turn_uid",
        TURN_TEXT_COLUMN,
    ],
    filter=ds.field(
        "turn_uid"
    ).isin(
        sample_turn_ids
    ),
)


turn_sample_df = (
    turn_table
    .to_pandas()
)


assert (
    "turn_uid"
    in turn_sample_df.columns
)

assert (
    TURN_TEXT_COLUMN
    in turn_sample_df.columns
)


turn_sample_df[
    "turn_uid"
] = (
    turn_sample_df[
        "turn_uid"
    ]
    .astype(str)
)


turn_sample_df[
    TURN_TEXT_COLUMN
] = (
    turn_sample_df[
        TURN_TEXT_COLUMN
    ]
    .fillna("")
    .astype(str)
)


assert (
    turn_sample_df[
        "turn_uid"
    ].is_unique
), (
    "turn_uid is not unique in the sampled "
    "session-turn index."
)


turn_lookup = (
    turn_sample_df
    .set_index(
        "turn_uid"
    )[
        TURN_TEXT_COLUMN
    ]
    .to_dict()
)


missing_turn_ids = (
    set(sample_turn_ids)
    -
    set(turn_lookup.keys())
)


assert not missing_turn_ids, (
    "R3 sample contains turn_uids missing "
    "from the R0 session-turn index.\n"
    +
    "\n".join(
        sorted(
            list(
                missing_turn_ids
            )
        )[:20]
    )
)


r3_sample[
    "turn_text"
] = (
    r3_sample[
        "turn_uid"
    ]
    .astype(str)
    .map(
        turn_lookup
    )
)


assert (
    r3_sample[
        "turn_text"
    ].notna()
    .all()
)


blank_turn_count = int(
    r3_sample[
        "turn_text"
    ]
    .str.strip()
    .eq("")
    .sum()
)


assert (
    blank_turn_count
    ==
    0
), (
    "Blank turn text detected in R3 pair-construction sample.\n"
    f"Blank rows: {blank_turn_count:,}"
)


print(
    "Turn rows resolved:",
    f"{len(turn_sample_df):,}",
)

print(
    "Missing turn UIDs:",
    0,
)

print(
    "Blank turn texts:",
    0,
)

print(
    "Turn text alignment:",
    "PASS",
)


turn_sample_df = (
    turn_table
    .to_pandas()
)


assert (
    "turn_uid"
    in turn_sample_df.columns
)

assert (
    TURN_TEXT_COLUMN
    in turn_sample_df.columns
)


turn_sample_df[
    "turn_uid"
] = (
    turn_sample_df[
        "turn_uid"
    ]
    .astype(str)
)


turn_sample_df[
    TURN_TEXT_COLUMN
] = (
    turn_sample_df[
        TURN_TEXT_COLUMN
    ]
    .fillna("")
    .astype(str)
)


assert (
    turn_sample_df[
        "turn_uid"
    ].is_unique
), (
    "turn_uid is not unique in the sampled "
    "session-turn index."
)


turn_lookup = (
    turn_sample_df
    .set_index(
        "turn_uid"
    )[
        TURN_TEXT_COLUMN
    ]
    .to_dict()
)


missing_turn_ids = (
    set(sample_turn_ids)
    -
    set(turn_lookup.keys())
)


assert not missing_turn_ids, (
    "R3 sample contains turn_uids missing "
    "from the R0 session-turn index.\n"
    +
    "\n".join(
        sorted(
            list(
                missing_turn_ids
            )
        )[:20]
    )
)


r3_sample[
    "turn_text"
] = (
    r3_sample[
        "turn_uid"
    ]
    .astype(str)
    .map(
        turn_lookup
    )
)


assert (
    r3_sample[
        "turn_text"
    ]
    .notna()
    .all()
)


# Blank turn text is a legitimate data-quality concern.
blank_turn_count = int(
    r3_sample[
        "turn_text"
    ]
    .str.strip()
    .eq("")
    .sum()
)


assert (
    blank_turn_count
    ==
    0
), (
    "Blank turn text detected in R3 pair-construction sample.\n"
    f"Blank rows: {blank_turn_count:,}"
)


print(
    "Missing turn UIDs:",
    0,
)

print(
    "Blank turn texts:",
    0,
)

print(
    "Turn text alignment:",
    "PASS",
)


# ==============================================================================
# 10. CANONICAL CROSS-ENCODER PAIRS
# ==============================================================================

r3_sample[
    "cross_encoder_pair"
] = list(
    zip(
        r3_sample[
            "objective_raw"
        ].astype(str),
        r3_sample[
            "turn_text"
        ].astype(str),
    )
)


assert (
    len(
        r3_sample[
            "cross_encoder_pair"
        ]
    )
    ==
    len(r3_sample)
)


assert all(
    (
        isinstance(pair, tuple)
        and
        len(pair) == 2
        and
        pair[0].strip() != ""
        and
        pair[1].strip() != ""
    )
    for pair in r3_sample[
        "cross_encoder_pair"
    ]
)


print("\n" + "=" * 90)
print("CROSS-ENCODER PAIR CONTRACT")
print("=" * 90)

print(
    "Pair format:",
    "(objective_text, turn_text)",
)

print(
    "Pairs constructed:",
    f"{len(r3_sample):,}",
)

print(
    "Empty objective texts:",
    0,
)

print(
    "Empty turn texts:",
    0,
)

print(
    "Pair construction:",
    "PASS",
)


# ==============================================================================
# 11. PAIR IDENTITY / DUPLICATE AUDIT
# ==============================================================================

pair_identity_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
]


duplicate_candidate_identity = int(
    r3_sample
    .duplicated(
        subset=pair_identity_columns
    )
    .sum()
)


assert (
    duplicate_candidate_identity
    ==
    0
), (
    "Duplicate candidate identities detected "
    "in pair-construction sample."
)


print("\n" + "=" * 90)
print("PAIR IDENTITY AUDIT")
print("=" * 90)

print(
    "Duplicate candidate identities:",
    duplicate_candidate_identity,
)

print(
    "Candidate identity:",
    "PASS",
)


# ==============================================================================
# 12. RESPONSE / SESSION CONSISTENCY
# ==============================================================================

response_session_counts = (
    r3_sample
    .groupby(
        "response_id",
        sort=False,
    )[
        "session_id"
    ]
    .nunique()
)


assert (
    response_session_counts
    .eq(1)
    .all()
), (
    "A response maps to multiple sessions "
    "inside the pair-construction sample."
)


print("\n" + "=" * 90)
print("RESPONSE / SESSION CONSISTENCY")
print("=" * 90)

print(
    "Responses spanning multiple sessions:",
    0,
)

print(
    "Response/session alignment:",
    "PASS",
)


# ==============================================================================
# 13. TARGET LEAKAGE AUDIT
# ==============================================================================

PAIR_FORBIDDEN_TERMS = {
    "target",
    "is_correct",
    "label",
    "y",
}


assert not (
    PAIR_FORBIDDEN_TERMS
    &
    set(
        r3_sample.columns
    )
), (
    "Target-bearing columns appeared during "
    "pair construction."
)


print("\n" + "=" * 90)
print("PAIR TARGET ISOLATION")
print("=" * 90)

print(
    "Target-bearing columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Pair target isolation:",
    "PASS",
)


# ==============================================================================
# 14. TOKENIZATION / TRUNCATION AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("TOKENIZATION / TRUNCATION AUDIT")
print("=" * 90)


tokenizer = cross_encoder.tokenizer


audit_pairs = (
    r3_sample[
        "cross_encoder_pair"
    ]
    .head(256)
    .tolist()
)


encoded = tokenizer(
    [pair[0] for pair in audit_pairs],
    [pair[1] for pair in audit_pairs],
    padding=False,
    truncation=True,
    max_length=CROSS_ENCODER_MAX_LENGTH,
    return_attention_mask=True,
)


audit_lengths = np.asarray(
    [
        sum(mask)
        for mask in encoded[
            "attention_mask"
        ]
    ],
    dtype=np.int32,
)


assert (
    len(audit_lengths)
    ==
    len(audit_pairs)
)


assert np.isfinite(
    audit_lengths
).all()


assert (
    audit_lengths.min()
    >
    0
)


assert (
    audit_lengths.max()
    <=
    CROSS_ENCODER_MAX_LENGTH
)


print(
    "Audited pairs:",
    f"{len(audit_pairs):,}",
)

print(
    "Minimum token length:",
    int(
        audit_lengths.min()
    ),
)

print(
    "Maximum token length:",
    int(
        audit_lengths.max()
    ),
)

print(
    "Configured max length:",
    CROSS_ENCODER_MAX_LENGTH,
)

print(
    "Tokenization:",
    "PASS",
)


# ==============================================================================
# 15. PAIR PREVIEW
# ==============================================================================

print("\n" + "=" * 90)
print("PAIR PREVIEW")
print("=" * 90)


preview_columns = [
    "response_id",
    "session_id",
    "objective_uid",
    "turn_uid",
    "role",
    "turn_index",
    "objective_raw",
    "turn_text",
]


print(
    r3_sample[
        preview_columns
    ]
    .head(10)
    .to_string(
        index=False,
        max_colwidth=120,
    )
)


# ==============================================================================
# 16. SAVE PAIR-CONSTRUCTION CONTRACT
# ==============================================================================

PAIR_CONTRACT = {
    "stage": "08_cross_encoder_reranking",
    "cell": 2,
    "upstream": "R3_frozen_candidate_union",
    "r3_candidate_rows": int(R3_ROWS),
    "audit_sample_rows": int(
        len(r3_sample)
    ),
    "objective_source": str(
        R0_OBJECTIVE_CATALOGUE
    ),
    "turn_source": str(
        R0_SESSION_TURN_INDEX
    ),
    "objective_text_column": "objective_raw",
    "turn_text_column": TURN_TEXT_COLUMN,
    "pair_format": [
        "objective_text",
        "turn_text",
    ],
    "cross_encoder_model": (
        CROSS_ENCODER_MODEL_NAME
    ),
    "max_length": int(
        CROSS_ENCODER_MAX_LENGTH
    ),
    "target_used": False,
    "target_columns": [],
    "objective_alignment": True,
    "turn_alignment": True,
    "blank_turn_texts": 0,
    "duplicate_candidate_identity": 0,
    "response_session_consistency": True,
    "tokenization_audit": True,
}


PAIR_CONTRACT_PATH = (
    CROSS_ENCODER_ROOT
    / "cross_encoder_pair_contract.json"
)


with open(
    PAIR_CONTRACT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        PAIR_CONTRACT,
        handle,
        indent=2,
    )


assert PAIR_CONTRACT_PATH.exists()


print("\n" + "=" * 90)
print("PAIR CONTRACT ARTIFACT")
print("=" * 90)

print(
    "Path:",
    PAIR_CONTRACT_PATH,
)

print(
    "Written:",
    True,
)


# ==============================================================================
# 17. CELL 2 FINAL GATE
# ==============================================================================

CROSS_ENCODER_PAIR_CONTRACT_READY = True
CROSS_ENCODER_CELL_2_READY = True


print("\n" + "=" * 90)
print("CELL 2 FINAL STATUS")
print("=" * 90)

print(
    "Objective alignment:",
    "PASS",
)

print(
    "Turn alignment:",
    "PASS",
)

print(
    "Pair construction:",
    "PASS",
)

print(
    "Candidate identity:",
    "PASS",
)

print(
    "Response/session consistency:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "Tokenization audit:",
    "PASS",
)

print(
    "Pair contract persisted:",
    PAIR_CONTRACT_PATH.exists(),
)

print(
    "CROSS_ENCODER_PAIR_CONTRACT_READY:",
    CROSS_ENCODER_PAIR_CONTRACT_READY,
)

print(
    "CROSS_ENCODER_CELL_2_READY:",
    CROSS_ENCODER_CELL_2_READY,
)


assert (
    CROSS_ENCODER_PAIR_CONTRACT_READY
    is True
)

assert (
    CROSS_ENCODER_CELL_2_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 2 — PAIR ALIGNMENT: PASS"
)
print("=" * 90)


# ==============================================================================
# 18. MEMORY CLEANUP
# ==============================================================================

if "r3_sample" in globals():
    del r3_sample

if "objective_df" in globals():
    del objective_df

if "turn_sample_df" in globals():
    del turn_sample_df

if "turn_table" in globals():
    del turn_table

if "objective_lookup" in globals():
    del objective_lookup

if "turn_lookup" in globals():
    del turn_lookup

if "encoded" in globals():
    del encoded

if "audit_lengths" in globals():
    del audit_lengths

if "audit_pairs" in globals():
    del audit_pairs

gc.collect()

print(
    "Cell 2 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 2 — CANDIDATE PAIR CONSTRUCTION + ALIGNMENT AUDIT

DEPENDENCY GATE
Cell 0 dependency: PASS
Cell 1 dependency: PASS

R0 ALIGNMENT INPUTS
Objective catalogue: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet
Session-turn index: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet
Objective catalogue: PASS
Session-turn index: PASS

R0 SCHEMA DISCOVERY
Objective columns: ['objective_uid', 'objective_raw']
Turn-index columns: ['session_id', 'turn_uid', 'turn_index', 'role', 'text_norm', 'relative_turn_position', 'previous_role', 'next_role', 'speaker_switch', 'time_since_previous_turn', 'elapsed_from_session_start', 'ordering_method', 'ordering_confidence', 'ordering_comparability']
Selected turn text column: text_norm

OBJECTIVE ALIGNMENT
Objective rows: 398
Unique objective UIDs: 398
Objective text non-blank: True
Objective ident

In [6]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 3 — RESUMABLE FULL-SCALE CROSS-ENCODER RERANKING
# ==============================================================================

from __future__ import annotations

import gc
import json
import os
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from sentence_transformers import CrossEncoder


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 3 — RESUMABLE FULL-SCALE CROSS-ENCODER RERANKING")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
), "Cell 0 must be executed first."

assert (
    CROSS_ENCODER_CELL_0_READY is True
), "Cell 0 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
), "Cell 1 must be executed first."

assert (
    CROSS_ENCODER_CELL_1_READY is True
), "Cell 1 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
), "Cell 2 must be executed first."

assert (
    CROSS_ENCODER_CELL_2_READY is True
), "Cell 2 dependency is not ready."


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print("Cell 0 dependency : PASS")
print("Cell 1 dependency : PASS")
print("Cell 2 dependency : PASS")


# ==============================================================================
# 2. PATHS
# ==============================================================================

R0_ROOT = (
    RETRIEVAL_ROOT
    / "R0_input"
)

R0_SESSION_TURN_INDEX = (
    R0_ROOT
    / "session_turn_index.parquet"
)

R3_CANDIDATE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

RERANK_ROOT = (
    CROSS_ENCODER_ROOT
    / "reranking"
)

LOOKUP_ROOT = (
    CROSS_ENCODER_ROOT
    / "turn_text_lookup"
)

RERANK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LOOKUP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# Persistent artifacts
TURN_LOOKUP_DB = (
    LOOKUP_ROOT
    / "turn_text_lookup.sqlite"
)

TURN_LOOKUP_MANIFEST = (
    LOOKUP_ROOT
    / "turn_text_lookup_manifest.json"
)

RERANK_OUTPUT = (
    RERANK_ROOT
    / "cross_encoder_scores.parquet"
)

RERANK_CHECKPOINT = (
    RERANK_ROOT
    / "cross_encoder_checkpoint.json"
)

RERANK_MANIFEST = (
    RERANK_ROOT
    / "cross_encoder_reranking_manifest.json"
)


print("\n" + "=" * 90)
print("RERANKING PATHS")
print("=" * 90)

print("R3 input       :", R3_CANDIDATE_PATH)
print("Turn lookup DB :", TURN_LOOKUP_DB)
print("Lookup manifest:", TURN_LOOKUP_MANIFEST)
print("Output         :", RERANK_OUTPUT)
print("Checkpoint     :", RERANK_CHECKPOINT)
print("Manifest       :", RERANK_MANIFEST)


# ==============================================================================
# 3. FROZEN INPUT CONTRACT
# ==============================================================================

assert R3_CANDIDATE_PATH.exists(), (
    f"Missing frozen R3 candidate:\n{R3_CANDIDATE_PATH}"
)

assert R0_SESSION_TURN_INDEX.exists(), (
    f"Missing R0 session-turn index:\n"
    f"{R0_SESSION_TURN_INDEX}"
)


R3_META = pq.ParquetFile(
    R3_CANDIDATE_PATH
).metadata

R3_TOTAL_ROWS = int(
    R3_META.num_rows
)

assert (
    R3_TOTAL_ROWS
    ==
    EXPECTED_R3_ROWS
), (
    "R3 row count changed.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {R3_TOTAL_ROWS:,}"
)


print("\n" + "=" * 90)
print("R3 INPUT CONTRACT")
print("=" * 90)

print(
    "Frozen R3 rows:",
    f"{R3_TOTAL_ROWS:,}",
)

print(
    "R3 input mutation:",
    "FORBIDDEN",
)

print(
    "R3 input contract:",
    "PASS",
)


# ==============================================================================
# 4. RERANKING CONFIGURATION
# ==============================================================================

RERANK_MODEL_NAME = (
    CROSS_ENCODER_MODEL_NAME
)

OUTER_CHUNK_ROWS = 5_000

MODEL_BATCH_SIZE = 32

CHECKPOINT_EVERY = 5

MAX_LENGTH = CROSS_ENCODER_MAX_LENGTH


print("\n" + "=" * 90)
print("RERANKING CONFIGURATION")
print("=" * 90)

print(
    "Model:",
    RERANK_MODEL_NAME,
)

print(
    "Outer chunk rows:",
    f"{OUTER_CHUNK_ROWS:,}",
)

print(
    "Model batch size:",
    MODEL_BATCH_SIZE,
)

print(
    "Checkpoint every:",
    CHECKPOINT_EVERY,
    "chunks",
)

print(
    "Max length:",
    MAX_LENGTH,
)


# ==============================================================================
# 5. CROSS-ENCODER LOAD
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER LOAD")
print("=" * 90)


if (
    "cross_encoder"
    not in globals()
):

    cross_encoder = CrossEncoder(
        RERANK_MODEL_NAME,
        max_length=MAX_LENGTH,
    )


print(
    "Model:",
    RERANK_MODEL_NAME,
)

print(
    "Device:",
    str(
        cross_encoder.model.device
    ),
)

print(
    "Model ready: PASS"
)


# ==============================================================================
# 6. OBJECTIVE LOOKUP
# ==============================================================================

R0_OBJECTIVE_CATALOGUE = (
    R0_ROOT
    / "objective_catalogue.parquet"
)

objective_df = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE,
    columns=[
        "objective_uid",
        "objective_raw",
    ],
)

assert (
    len(objective_df)
    ==
    EXPECTED_R3_OBJECTIVES
)

assert (
    objective_df["objective_uid"]
    .is_unique
)

objective_lookup = (
    objective_df
    .set_index(
        "objective_uid"
    )[
        "objective_raw"
    ]
    .astype(str)
    .to_dict()
)

del objective_df

print(
    "Objective lookup:",
    f"{len(objective_lookup):,} objectives",
)


# ==============================================================================
# 7. BUILD DISK-BASED TURN TEXT LOOKUP
# ==============================================================================

print("\n" + "=" * 90)
print("TURN TEXT LOOKUP")
print("=" * 90)


def build_turn_lookup_database():

    if TURN_LOOKUP_DB.exists():
        print(
            "Existing lookup database found."
        )

        if TURN_LOOKUP_MANIFEST.exists():

            with open(
                TURN_LOOKUP_MANIFEST,
                "r",
                encoding="utf-8",
            ) as handle:

                existing_manifest = json.load(
                    handle
                )

            if (
                existing_manifest.get(
                    "status"
                )
                ==
                "COMPLETE"
            ):

                print(
                    "Lookup manifest: COMPLETE"
                )

                return

        print(
            "Lookup exists without a valid COMPLETE manifest."
        )

        print(
            "Removing incomplete lookup."
        )

        TURN_LOOKUP_DB.unlink(
            missing_ok=True
        )


    print(
        "\nBuilding disk-based turn text lookup."
    )

    print(
        "Source:",
        R0_SESSION_TURN_INDEX,
    )

    lookup_tmp = (
        LOOKUP_ROOT
        / "turn_text_lookup.sqlite.tmp"
    )

    lookup_tmp.unlink(
        missing_ok=True
    )

    conn = sqlite3.connect(
        lookup_tmp
    )

    try:

        conn.execute(
            """
            PRAGMA journal_mode = WAL;
            """
        )

        conn.execute(
            """
            PRAGMA synchronous = NORMAL;
            """
        )

        conn.execute(
            """
            CREATE TABLE turn_text (
                turn_uid TEXT PRIMARY KEY,
                text_norm TEXT NOT NULL
            );
            """
        )

        conn.commit()


        scanner = ds.dataset(
            R0_SESSION_TURN_INDEX,
            format="parquet",
        ).scanner(
            columns=[
                "turn_uid",
                "text_norm",
            ],
            batch_size=100_000,
        )


        inserted = 0

        for batch_index, batch in enumerate(
            scanner.to_batches(),
            start=1,
        ):

            batch_df = (
                batch
                .to_pandas()
            )

            batch_df[
                "turn_uid"
            ] = (
                batch_df[
                    "turn_uid"
                ]
                .astype(str)
            )

            batch_df[
                "text_norm"
            ] = (
                batch_df[
                    "text_norm"
                ]
                .fillna("")
                .astype(str)
            )

            rows = list(
                zip(
                    batch_df[
                        "turn_uid"
                    ],
                    batch_df[
                        "text_norm"
                    ],
                )
            )

            conn.executemany(
                """
                INSERT OR REPLACE INTO turn_text
                (
                    turn_uid,
                    text_norm
                )
                VALUES (?, ?)
                """,
                rows,
            )

            inserted += len(rows)

            if (
                batch_index % 10
                ==
                0
            ):

                conn.commit()

                print(
                    f"Lookup rows indexed: "
                    f"{inserted:,}"
                )

            del batch_df
            del rows

            gc.collect()


        conn.commit()

        conn.execute(
            """
            CREATE INDEX IF NOT EXISTS
            idx_turn_uid
            ON turn_text(turn_uid);
            """
        )

        conn.commit()

    finally:

        conn.close()


    os.replace(
        lookup_tmp,
        TURN_LOOKUP_DB,
    )


    lookup_conn = sqlite3.connect(
        TURN_LOOKUP_DB
    )

    try:

        count = int(
            lookup_conn.execute(
                """
                SELECT COUNT(*)
                FROM turn_text
                """
            ).fetchone()[0]
        )

    finally:

        lookup_conn.close()


    lookup_manifest = {
        "artifact": "turn_text_lookup",
        "status": "COMPLETE",
        "source": str(
            R0_SESSION_TURN_INDEX
        ),
        "rows": count,
        "text_column": "text_norm",
        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }


    with open(
        TURN_LOOKUP_MANIFEST,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            lookup_manifest,
            handle,
            indent=2,
        )


    print(
        "Turn lookup rows:",
        f"{count:,}",
    )

    print(
        "Turn lookup manifest:",
        "COMPLETE",
    )


build_turn_lookup_database()


# ==============================================================================
# 8. VERIFY TURN LOOKUP
# ==============================================================================

with open(
    TURN_LOOKUP_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    lookup_manifest = json.load(
        handle
    )


assert (
    lookup_manifest.get(
        "status"
    )
    ==
    "COMPLETE"
)

assert (
    TURN_LOOKUP_DB.exists()
)


lookup_conn = sqlite3.connect(
    TURN_LOOKUP_DB
)

try:

    lookup_count = int(
        lookup_conn.execute(
            """
            SELECT COUNT(*)
            FROM turn_text
            """
        ).fetchone()[0]
    )

finally:

    lookup_conn.close()


assert (
    lookup_count
    ==
    6_139_854
), (
    "Unexpected turn lookup population.\n"
    f"Expected: 6,139,854\n"
    f"Observed: {lookup_count:,}"
)


print("\nTurn lookup verification: PASS")
print(
    "Turn rows:",
    f"{lookup_count:,}",
)


# ==============================================================================
# 9. CHECKPOINT HELPERS
# ==============================================================================

def load_checkpoint():

    if not RERANK_CHECKPOINT.exists():

        return {
            "status": "NOT_STARTED",
            "completed_rows": 0,
            "completed_chunks": 0,
        }

    with open(
        RERANK_CHECKPOINT,
        "r",
        encoding="utf-8",
    ) as handle:

        checkpoint = json.load(
            handle
        )

    return checkpoint


def save_checkpoint(
    completed_rows,
    completed_chunks,
    status,
):

    checkpoint = {
        "status": status,
        "completed_rows": int(
            completed_rows
        ),
        "completed_chunks": int(
            completed_chunks
        ),
        "total_rows": int(
            R3_TOTAL_ROWS
        ),
        "updated_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }

    tmp_path = Path(
        str(
            RERANK_CHECKPOINT
        )
        + ".tmp"
    )

    with open(
        tmp_path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            checkpoint,
            handle,
            indent=2,
        )

    os.replace(
        tmp_path,
        RERANK_CHECKPOINT,
    )


# ==============================================================================
# 10. RESUME STATE
# ==============================================================================

checkpoint = load_checkpoint()

completed_rows = int(
    checkpoint.get(
        "completed_rows",
        0,
    )
)

completed_chunks = int(
    checkpoint.get(
        "completed_chunks",
        0,
    )
)

checkpoint_status = checkpoint.get(
    "status",
    "NOT_STARTED",
)


print("\n" + "=" * 90)
print("RESUME STATE")
print("=" * 90)

print(
    "Checkpoint status:",
    checkpoint_status,
)

print(
    "Completed rows:",
    f"{completed_rows:,}",
)

print(
    "Completed chunks:",
    completed_chunks,
)

print(
    "Remaining rows:",
    f"{R3_TOTAL_ROWS - completed_rows:,}",
)


# ==============================================================================
# 11. OUTPUT CONTRACT
# ==============================================================================

OUTPUT_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
]


def output_is_compatible():

    if not RERANK_OUTPUT.exists():
        return False

    parquet_file = pq.ParquetFile(
        RERANK_OUTPUT
    )

    columns = list(
        parquet_file
        .schema_arrow
        .names
    )

    return (
        columns
        ==
        OUTPUT_COLUMNS
    )


if completed_rows > 0:

    assert RERANK_OUTPUT.exists(), (
        "Checkpoint says rows are completed, "
        "but reranking output is missing."
    )

    assert output_is_compatible(), (
        "Existing reranking output schema is incompatible."
    )


# ==============================================================================
# 12. PROCESS ONE CHUNK
# ==============================================================================

def resolve_turn_texts(
    turn_uids,
):

    conn = sqlite3.connect(
        TURN_LOOKUP_DB
    )

    try:

        unique_ids = list(
            dict.fromkeys(
                str(x)
                for x in turn_uids
            )
        )

        result = {}

        # SQLite parameter limit is avoided by chunks.
        lookup_batch_size = 800

        for start in range(
            0,
            len(unique_ids),
            lookup_batch_size,
        ):

            ids = unique_ids[
                start:
                start
                + lookup_batch_size
            ]

            placeholders = ",".join(
                "?"
                for _ in ids
            )

            query = f"""
                SELECT
                    turn_uid,
                    text_norm
                FROM turn_text
                WHERE turn_uid IN (
                    {placeholders}
                )
            """

            rows = conn.execute(
                query,
                ids,
            ).fetchall()

            for uid, text in rows:

                result[
                    uid
                ] = text

        return result

    finally:

        conn.close()


def score_chunk(
    chunk_df,
):

    chunk_df = (
        chunk_df
        .copy()
    )


    chunk_df[
        "objective_raw"
    ] = (
        chunk_df[
            "objective_uid"
        ]
        .map(
            objective_lookup
        )
    )


    assert (
        chunk_df[
            "objective_raw"
        ]
        .notna()
        .all()
    )


    turn_lookup = resolve_turn_texts(
        chunk_df[
            "turn_uid"
        ]
        .astype(str)
        .tolist()
    )


    chunk_df[
        "turn_text"
    ] = (
        chunk_df[
            "turn_uid"
        ]
        .astype(str)
        .map(
            turn_lookup
        )
    )


    assert (
        chunk_df[
            "turn_text"
        ]
        .notna()
        .all()
    ), (
        "One or more turn_uids could not be resolved."
    )


    assert (
        chunk_df[
            "turn_text"
        ]
        .astype(str)
        .str.strip()
        .ne("")
        .all()
    ), (
        "Blank turn text detected."
    )


    pairs = list(
        zip(
            chunk_df[
                "objective_raw"
            ].astype(str),
            chunk_df[
                "turn_text"
            ].astype(str),
        )
    )


    scores = cross_encoder.predict(
        pairs,
        batch_size=MODEL_BATCH_SIZE,
        show_progress_bar=False,
    )


    scores = np.asarray(
        scores,
        dtype=np.float32,
    ).reshape(-1)


    assert (
        len(scores)
        ==
        len(chunk_df)
    )


    assert np.isfinite(
        scores
    ).all(), (
        "Non-finite Cross-Encoder scores detected."
    )


    result = chunk_df[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
            "turn_uid",
            "role",
            "turn_index",
        ]
    ].copy()


    result[
        "cross_encoder_score"
    ] = scores


    return result


# ==============================================================================
# 13. RESUMABLE RERANKING LOOP
# ==============================================================================

print("\n" + "=" * 90)
print("FULL-SCALE RERANKING")
print("=" * 90)

print(
    "Total rows:",
    f"{R3_TOTAL_ROWS:,}",
)

print(
    "Already completed:",
    f"{completed_rows:,}",
)

print(
    "Remaining:",
    f"{R3_TOTAL_ROWS - completed_rows:,}",
)


if completed_rows == 0:

    if RERANK_OUTPUT.exists():

        print(
            "No checkpoint exists."
        )

        print(
            "Removing stale reranking output."
        )

        RERANK_OUTPUT.unlink()


    output_writer = None

else:

    output_writer = None


r3_dataset = ds.dataset(
    R3_CANDIDATE_PATH,
    format="parquet",
)


r3_scanner = r3_dataset.scanner(
    columns=[
        "response_id",
        "session_id",
        "objective_uid",
        "fold",
        "turn_uid",
        "role",
        "turn_index",
    ],
    batch_size=OUTER_CHUNK_ROWS,
)


processed_rows = 0
processed_chunks = 0

start_time = time.time()


for batch_index, batch in enumerate(
    r3_scanner.to_batches(),
):

    chunk_start_row = (
        processed_rows
    )

    chunk_df = (
        batch
        .to_pandas()
    )


    # --------------------------------------------------------------------------
    # RESUME
    # --------------------------------------------------------------------------

    if (
        chunk_start_row
        +
        len(chunk_df)
        <=
        completed_rows
    ):

        processed_rows += len(
            chunk_df
        )

        continue


    if chunk_start_row < completed_rows:

        skip_inside_chunk = (
            completed_rows
            -
            chunk_start_row
        )

        chunk_df = (
            chunk_df
            .iloc[
                skip_inside_chunk:
            ]
            .copy()
        )


    # --------------------------------------------------------------------------
    # SCORE
    # --------------------------------------------------------------------------

    chunk_result = score_chunk(
        chunk_df
    )


    # --------------------------------------------------------------------------
    # OUTPUT WRITER
    # --------------------------------------------------------------------------

    if output_writer is None:

        output_schema = pa.Table.from_pandas(
            chunk_result,
            preserve_index=False,
        ).schema

        output_writer = pq.ParquetWriter(
            RERANK_OUTPUT,
            output_schema,
            compression="zstd",
        )


    output_table = pa.Table.from_pandas(
        chunk_result,
        preserve_index=False,
    )


    output_writer.write_table(
        output_table
    )


    # --------------------------------------------------------------------------
    # UPDATE STATE
    # --------------------------------------------------------------------------

    processed_rows += len(
        chunk_result
    )

    processed_chunks += 1


    if (
        processed_chunks
        %
        CHECKPOINT_EVERY
        ==
        0
    ):

        save_checkpoint(
            completed_rows=processed_rows,
            completed_chunks=processed_chunks,
            status="RUNNING",
        )


        elapsed = (
            time.time()
            -
            start_time
        )

        rate = (
            processed_rows
            /
            elapsed
            if elapsed > 0
            else 0.0
        )

        remaining = (
            R3_TOTAL_ROWS
            -
            processed_rows
        )

        eta_hours = (
            remaining / rate / 3600
            if rate > 0
            else float("inf")
        )


        print(
            f"Processed "
            f"{processed_rows:,}/"
            f"{R3_TOTAL_ROWS:,} "
            f"("
            f"{processed_rows / R3_TOTAL_ROWS * 100:.2f}%"
            f") | "
            f"rate={rate:.1f} rows/s | "
            f"ETA={eta_hours:.2f}h"
        )


    # --------------------------------------------------------------------------
    # MEMORY CLEANUP
    # --------------------------------------------------------------------------

    del chunk_df
    del chunk_result
    del output_table
    del batch

    gc.collect()


# ==============================================================================
# 14. CLOSE OUTPUT WRITER
# ==============================================================================

if output_writer is not None:

    output_writer.close()

    output_writer = None


# ==============================================================================
# 15. FINAL COMPLETION CHECK
# ==============================================================================

assert (
    processed_rows
    ==
    R3_TOTAL_ROWS
), (
    "Reranking did not process the expected "
    "number of R3 rows.\n"
    f"Expected: {R3_TOTAL_ROWS:,}\n"
    f"Observed: {processed_rows:,}"
)


assert RERANK_OUTPUT.exists(), (
    "Reranking output was not created."
)


save_checkpoint(
    completed_rows=R3_TOTAL_ROWS,
    completed_chunks=processed_chunks,
    status="COMPLETE",
)


# ==============================================================================
# 16. OUTPUT VALIDATION
# ==============================================================================

rerank_parquet = pq.ParquetFile(
    RERANK_OUTPUT
)

rerank_rows = int(
    rerank_parquet
    .metadata
    .num_rows
)

rerank_columns = list(
    rerank_parquet
    .schema_arrow
    .names
)


assert (
    rerank_rows
    ==
    R3_TOTAL_ROWS
), (
    "Reranking output row count mismatch."
)

assert (
    rerank_columns
    ==
    OUTPUT_COLUMNS
), (
    "Reranking output schema mismatch."
)


print("\n" + "=" * 90)
print("RERANKING OUTPUT")
print("=" * 90)

print(
    "Rows:",
    f"{rerank_rows:,}",
)

print(
    "Columns:",
    rerank_columns,
)

print(
    "Schema:",
    "PASS",
)

print(
    "Population:",
    "PASS",
)


# ==============================================================================
# 17. RERANKING MANIFEST
# ==============================================================================

RERANK_MANIFEST_DATA = {
    "artifact": "cross_encoder_scores",
    "status": "COMPLETE",
    "model": RERANK_MODEL_NAME,
    "upstream_artifact": "r3_candidate_union",
    "upstream_rows": int(
        R3_TOTAL_ROWS
    ),
    "output_rows": int(
        rerank_rows
    ),
    "outer_chunk_rows": int(
        OUTER_CHUNK_ROWS
    ),
    "model_batch_size": int(
        MODEL_BATCH_SIZE
    ),
    "max_length": int(
        MAX_LENGTH
    ),
    "target_used": False,
    "output_path": str(
        RERANK_OUTPUT
    ),
    "checkpoint_path": str(
        RERANK_CHECKPOINT
    ),
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}


with open(
    RERANK_MANIFEST,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        RERANK_MANIFEST_DATA,
        handle,
        indent=2,
    )


assert RERANK_MANIFEST.exists()


# ==============================================================================
# 18. FINAL STATUS
# ==============================================================================

CROSS_ENCODER_RERANKING_COMPLETE = True
CROSS_ENCODER_CELL_3_READY = True


print("\n" + "=" * 90)
print("RERANKING FINAL STATUS")
print("=" * 90)

print(
    "Processed rows:",
    f"{processed_rows:,}",
)

print(
    "Expected rows:",
    f"{R3_TOTAL_ROWS:,}",
)

print(
    "Checkpoint status:",
    "COMPLETE",
)

print(
    "Output artifact:",
    RERANK_OUTPUT,
)

print(
    "Manifest:",
    RERANK_MANIFEST,
)

print(
    "Target used:",
    False,
)

print(
    "Reranking complete:",
    CROSS_ENCODER_RERANKING_COMPLETE,
)

print(
    "CROSS_ENCODER_CELL_3_READY:",
    CROSS_ENCODER_CELL_3_READY,
)


assert (
    CROSS_ENCODER_RERANKING_COMPLETE
    is True
)

assert (
    CROSS_ENCODER_CELL_3_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 3 — FULL RERANKING: PASS"
)
print("=" * 90)


# ==============================================================================
# 19. MEMORY CLEANUP
# ==============================================================================

if "objective_lookup" in globals():
    del objective_lookup

if "r3_dataset" in globals():
    del r3_dataset

if "r3_scanner" in globals():
    del r3_scanner

if "objective_df" in globals():
    del objective_df

gc.collect()

print(
    "Cell 3 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 3 — RESUMABLE FULL-SCALE CROSS-ENCODER RERANKING

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS

RERANKING PATHS
R3 input       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_candidate_union.parquet
Turn lookup DB : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\turn_text_lookup\turn_text_lookup.sqlite
Lookup manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\turn_text_lookup\turn_text_lookup_manifest.json
Output         : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_scores.parquet
Checkpoint     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_checkpoint.json
Manifest       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_

In [8]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 4 — RERANKING OUTPUT RELOAD + EXACT IDENTITY AUDIT
# ==============================================================================

from __future__ import annotations

import gc
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 4 — RERANKING OUTPUT RELOAD + EXACT IDENTITY AUDIT")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
), (
    "Cell 0 must be executed before Cell 4."
)

assert (
    CROSS_ENCODER_CELL_0_READY
    is True
), (
    "Cross-Encoder Cell 0 dependency is not ready."
)


assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
), (
    "Cell 1 must be executed before Cell 4."
)

assert (
    CROSS_ENCODER_CELL_1_READY
    is True
), (
    "Cross-Encoder Cell 1 dependency is not ready."
)


assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
), (
    "Cell 2 must be executed before Cell 4."
)

assert (
    CROSS_ENCODER_CELL_2_READY
    is True
), (
    "Cross-Encoder Cell 2 dependency is not ready."
)


assert (
    "CROSS_ENCODER_CELL_3_READY" in globals()
), (
    "Cell 3 must be executed before Cell 4."
)

assert (
    CROSS_ENCODER_CELL_3_READY
    is True
), (
    "Cross-Encoder Cell 3 dependency is not ready."
)


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print("Cell 0 dependency : PASS")
print("Cell 1 dependency : PASS")
print("Cell 2 dependency : PASS")
print("Cell 3 dependency : PASS")


# ==============================================================================
# 2. PATH CONTRACT
# ==============================================================================

R3_CANDIDATE_PATH = (
    R3_FREEZE_ROOT
    / "r3_candidate_union.parquet"
)

RERANK_ROOT = (
    CROSS_ENCODER_ROOT
    / "reranking"
)

RERANK_OUTPUT = (
    RERANK_ROOT
    / "cross_encoder_scores.parquet"
)

RERANK_CHECKPOINT = (
    RERANK_ROOT
    / "cross_encoder_checkpoint.json"
)

RERANK_MANIFEST = (
    RERANK_ROOT
    / "cross_encoder_reranking_manifest.json"
)


print("\n" + "=" * 90)
print("ARTIFACT PATHS")
print("=" * 90)

print(
    "R3 candidate:",
    R3_CANDIDATE_PATH,
)

print(
    "Reranking output:",
    RERANK_OUTPUT,
)

print(
    "Checkpoint:",
    RERANK_CHECKPOINT,
)

print(
    "Manifest:",
    RERANK_MANIFEST,
)


assert R3_CANDIDATE_PATH.exists(), (
    "Frozen R3 candidate is missing."
)

assert RERANK_OUTPUT.exists(), (
    "Cross-Encoder output is missing."
)

assert RERANK_CHECKPOINT.exists(), (
    "Cross-Encoder checkpoint is missing."
)

assert RERANK_MANIFEST.exists(), (
    "Cross-Encoder reranking manifest is missing."
)


print(
    "Artifact existence:",
    "PASS",
)


# ==============================================================================
# 3. LOAD CHECKPOINT
# ==============================================================================

with open(
    RERANK_CHECKPOINT,
    "r",
    encoding="utf-8",
) as handle:

    rerank_checkpoint = json.load(
        handle
    )


assert (
    rerank_checkpoint.get(
        "status"
    )
    ==
    "COMPLETE"
), (
    "Reranking checkpoint is not COMPLETE."
)


checkpoint_rows = int(
    rerank_checkpoint.get(
        "completed_rows",
        -1,
    )
)


assert (
    checkpoint_rows
    ==
    EXPECTED_R3_ROWS
), (
    "Checkpoint population mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {checkpoint_rows:,}"
)


print("\n" + "=" * 90)
print("CHECKPOINT CONTRACT")
print("=" * 90)

print(
    "Status:",
    rerank_checkpoint.get(
        "status"
    ),
)

print(
    "Completed rows:",
    f"{checkpoint_rows:,}",
)

print(
    "Checkpoint:",
    "PASS",
)


# ==============================================================================
# 4. LOAD MANIFEST
# ==============================================================================

with open(
    RERANK_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    rerank_manifest = json.load(
        handle
    )


assert (
    rerank_manifest.get(
        "status"
    )
    ==
    "COMPLETE"
), (
    "Reranking manifest status is not COMPLETE."
)


assert (
    rerank_manifest.get(
        "model"
    )
    ==
    CROSS_ENCODER_MODEL_NAME
), (
    "Reranking model does not match Cell 1 contract."
)


assert (
    int(
        rerank_manifest.get(
            "upstream_rows",
            -1,
        )
    )
    ==
    EXPECTED_R3_ROWS
)


assert (
    int(
        rerank_manifest.get(
            "output_rows",
            -1,
        )
    )
    ==
    EXPECTED_R3_ROWS
)


assert (
    rerank_manifest.get(
        "target_used"
    )
    is False
), (
    "Reranking manifest indicates target usage."
)


print("\n" + "=" * 90)
print("RERANKING MANIFEST")
print("=" * 90)

print(
    "JSON valid:",
    "PASS",
)

print(
    "Status:",
    rerank_manifest.get(
        "status"
    ),
)

print(
    "Model:",
    rerank_manifest.get(
        "model"
    ),
)

print(
    "Upstream rows:",
    f"{int(rerank_manifest['upstream_rows']):,}",
)

print(
    "Output rows:",
    f"{int(rerank_manifest['output_rows']):,}",
)

print(
    "Target used:",
    rerank_manifest.get(
        "target_used"
    ),
)


# ==============================================================================
# 5. PARQUET METADATA
# ==============================================================================

r3_parquet = pq.ParquetFile(
    R3_CANDIDATE_PATH
)

rerank_parquet = pq.ParquetFile(
    RERANK_OUTPUT
)


R3_ROWS_OBSERVED = int(
    r3_parquet
    .metadata
    .num_rows
)

RERANK_ROWS_OBSERVED = int(
    rerank_parquet
    .metadata
    .num_rows
)


print("\n" + "=" * 90)
print("PARQUET POPULATION")
print("=" * 90)

print(
    "R3 rows:",
    f"{R3_ROWS_OBSERVED:,}",
)

print(
    "Reranking rows:",
    f"{RERANK_ROWS_OBSERVED:,}",
)

print(
    "Expected:",
    f"{EXPECTED_R3_ROWS:,}",
)


assert (
    R3_ROWS_OBSERVED
    ==
    EXPECTED_R3_ROWS
)

assert (
    RERANK_ROWS_OBSERVED
    ==
    EXPECTED_R3_ROWS
)

print(
    "Population contract:",
    "PASS",
)


# ==============================================================================
# 6. SCHEMA CONTRACT
# ==============================================================================

R3_IDENTITY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
]


EXPECTED_RERANK_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
]


R3_COLUMNS_OBSERVED = list(
    r3_parquet
    .schema_arrow
    .names
)

RERANK_COLUMNS_OBSERVED = list(
    rerank_parquet
    .schema_arrow
    .names
)


print("\n" + "=" * 90)
print("SCHEMA CONTRACT")
print("=" * 90)

print(
    "R3 columns:",
    R3_COLUMNS_OBSERVED,
)

print(
    "Reranking columns:",
    RERANK_COLUMNS_OBSERVED,
)


for column in R3_IDENTITY_COLUMNS:

    assert (
        column
        in R3_COLUMNS_OBSERVED
    ), (
        f"Missing {column} from R3."
    )

    assert (
        column
        in RERANK_COLUMNS_OBSERVED
    ), (
        f"Missing {column} from reranking output."
    )


assert (
    RERANK_COLUMNS_OBSERVED
    ==
    EXPECTED_RERANK_COLUMNS
), (
    "Unexpected reranking output schema."
)


print(
    "Schema contract:",
    "PASS",
)


# ==============================================================================
# 7. STREAMING EXACT IDENTITY AUDIT
# ==============================================================================
#
# IMPORTANT:
# R3 and reranking parquet files do NOT necessarily have identical
# row-group / batch boundaries.
#
# Therefore we must NOT compare:
#
#     next(R3 scanner batch)
#     vs
#     next(reranking scanner batch)
#
# by assuming equal row counts.
#
# This implementation creates logical fixed-size windows independent
# of the underlying Parquet row-group boundaries.
# ==============================================================================

print("\n" + "=" * 90)
print("EXACT R3 → RERANKING IDENTITY AUDIT")
print("=" * 90)

print(
    "Audit mode:",
    "ROW-GROUP-INDEPENDENT STREAMING",
)

print(
    "No full 2.48M-row DataFrame will be materialized.",
)


AUDIT_WINDOW_ROWS = 50_000


# ------------------------------------------------------------------------------
# Helper: accumulate scanner batches into an exact logical window
# ------------------------------------------------------------------------------

def take_logical_window(
    batch_iterator,
    pending_df,
    target_rows,
    columns,
):
    """
    Consume arbitrary Arrow scanner batches until exactly target_rows
    are available, unless the iterator is exhausted.

    Returns:
        window_df
        remaining_pending_df
        exhausted
    """

    pieces = []

    if (
        pending_df is not None
        and
        len(pending_df) > 0
    ):

        pieces.append(
            pending_df
        )

        pending_df = None


    current_rows = sum(
        len(piece)
        for piece in pieces
    )


    while current_rows < target_rows:

        try:

            batch = next(
                batch_iterator
            )

        except StopIteration:

            break


        batch_df = (
            batch
            .to_pandas()
        )


        pieces.append(
            batch_df
        )

        current_rows += len(
            batch_df
        )


        del batch


    if not pieces:

        return (
            None,
            None,
            True,
        )


    combined = pd.concat(
        pieces,
        ignore_index=True,
    )


    if (
        len(combined)
        <=
        target_rows
    ):

        return (
            combined,
            None,
            False,
        )


    window_df = (
        combined
        .iloc[
            :target_rows
        ]
        .copy()
    )


    remaining_df = (
        combined
        .iloc[
            target_rows:
        ]
        .copy()
    )


    return (
        window_df,
        remaining_df,
        False,
    )


# ------------------------------------------------------------------------------
# Dataset scanners
# ------------------------------------------------------------------------------

r3_dataset = ds.dataset(
    R3_CANDIDATE_PATH,
    format="parquet",
)

rerank_dataset = ds.dataset(
    RERANK_OUTPUT,
    format="parquet",
)


r3_scanner = r3_dataset.scanner(
    columns=R3_IDENTITY_COLUMNS,
    batch_size=AUDIT_WINDOW_ROWS,
)


rerank_scanner = rerank_dataset.scanner(
    columns=R3_IDENTITY_COLUMNS
    + [
        "cross_encoder_score",
    ],
    batch_size=AUDIT_WINDOW_ROWS,
)


r3_batches = iter(
    r3_scanner.to_batches()
)

rerank_batches = iter(
    rerank_scanner.to_batches()
)


r3_pending = None
rerank_pending = None


identity_rows_checked = 0
score_rows_checked = 0
audit_windows = 0


while True:

    (
        r3_window,
        r3_pending,
        r3_exhausted,
    ) = take_logical_window(
        r3_batches,
        r3_pending,
        AUDIT_WINDOW_ROWS,
        R3_IDENTITY_COLUMNS,
    )


    (
        rerank_window,
        rerank_pending,
        rerank_exhausted,
    ) = take_logical_window(
        rerank_batches,
        rerank_pending,
        AUDIT_WINDOW_ROWS,
        R3_IDENTITY_COLUMNS
        + [
            "cross_encoder_score",
        ],
    )


    # --------------------------------------------------------------------------
    # BOTH STREAMS FINISHED
    # --------------------------------------------------------------------------

    if (
        r3_window is None
        and
        rerank_window is None
    ):

        break


    assert (
        r3_window is not None
        and
        rerank_window is not None
    ), (
        "R3 and reranking streams have different total populations."
    )


    # --------------------------------------------------------------------------
    # WINDOW ROW COUNT
    # --------------------------------------------------------------------------

    assert (
        len(r3_window)
        ==
        len(rerank_window)
    ), (
        "Logical audit window row count mismatch.\n"
        f"R3 rows: {len(r3_window):,}\n"
        f"Rerank rows: {len(rerank_window):,}\n"
        f"Rows already checked: "
        f"{identity_rows_checked:,}"
    )


    # --------------------------------------------------------------------------
    # EXACT IDENTITY + ORDER
    # --------------------------------------------------------------------------

    for column in R3_IDENTITY_COLUMNS:

        left = (
            r3_window[
                column
            ]
            .astype(str)
            .to_numpy()
        )


        right = (
            rerank_window[
                column
            ]
            .astype(str)
            .to_numpy()
        )


        assert np.array_equal(
            left,
            right,
        ), (
            "R3 → reranking identity/order mismatch.\n"
            f"Column: {column}\n"
            f"Audit window: {audit_windows + 1}\n"
            f"Rows already checked: "
            f"{identity_rows_checked:,}"
        )


    # --------------------------------------------------------------------------
    # SCORE CONTRACT
    # --------------------------------------------------------------------------

    scores = pd.to_numeric(
        rerank_window[
            "cross_encoder_score"
        ],
        errors="coerce",
    )


    assert (
        scores.notna().all()
    ), (
        "NaN/non-numeric Cross-Encoder score detected."
    )


    score_array = (
        scores
        .to_numpy(
            dtype=np.float32
        )
    )


    assert np.isfinite(
        score_array
    ).all(), (
        "Non-finite Cross-Encoder score detected."
    )


    # --------------------------------------------------------------------------
    # UPDATE COUNTERS
    # --------------------------------------------------------------------------

    window_rows = len(
        r3_window
    )


    identity_rows_checked += (
        window_rows
    )

    score_rows_checked += (
        window_rows
    )

    audit_windows += 1


    if (
        audit_windows % 10
        ==
        0
    ):

        print(
            "Checked:",
            f"{identity_rows_checked:,}/"
            f"{EXPECTED_R3_ROWS:,}",
            "| windows:",
            audit_windows,
        )


    # --------------------------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------------------------

    del r3_window
    del rerank_window
    del scores
    del score_array

    gc.collect()


# ==============================================================================
# FINAL POPULATION ASSERTIONS
# ==============================================================================

assert (
    identity_rows_checked
    ==
    EXPECTED_R3_ROWS
), (
    "Not all R3 rows were identity-audited.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {identity_rows_checked:,}"
)


assert (
    score_rows_checked
    ==
    EXPECTED_R3_ROWS
), (
    "Not all reranking scores were audited.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {score_rows_checked:,}"
)


print(
    "Identity rows checked:",
    f"{identity_rows_checked:,}",
)

print(
    "Score rows checked:",
    f"{score_rows_checked:,}",
)

print(
    "Audit windows:",
    f"{audit_windows:,}",
)

print(
    "Identity/order alignment:",
    "PASS",
)

print(
    "Score finiteness:",
    "PASS",
)

# ==============================================================================
# 8. SCORE SEMANTICS AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("CROSS-ENCODER SCORE CONTRACT")
print("=" * 90)

print(
    "Score column:",
    "cross_encoder_score",
)

print(
    "Score type:",
    "raw Cross-Encoder relevance score",
)

print(
    "Probability calibration:",
    "NOT APPLIED",
)

print(
    "[0,1] range requirement:",
    "NOT APPLICABLE",
)

print(
    "NaN:",
    0,
)

print(
    "Inf:",
    0,
)


# ==============================================================================
# 9. SCORE DISTRIBUTION SAMPLE
# ==============================================================================

SCORE_SAMPLE_ROWS = 100_000


score_sample_table = (
    rerank_dataset
    .to_table(
        columns=[
            "cross_encoder_score",
        ],
        filter=None,
    )
)


score_sample_df = (
    score_sample_table
    .to_pandas()
)


if len(score_sample_df) > SCORE_SAMPLE_ROWS:

    score_sample_df = (
        score_sample_df
        .head(
            SCORE_SAMPLE_ROWS
        )
        .copy()
    )


score_sample_values = pd.to_numeric(
    score_sample_df[
        "cross_encoder_score"
    ],
    errors="coerce",
).to_numpy(
    dtype=np.float64
)


assert np.isfinite(
    score_sample_values
).all()


print(
    "Score sample rows:",
    f"{len(score_sample_values):,}",
)

print(
    "Score minimum:",
    float(
        score_sample_values.min()
    ),
)

print(
    "Score maximum:",
    float(
        score_sample_values.max()
    ),
)

print(
    "Score mean:",
    float(
        score_sample_values.mean()
    ),
)

print(
    "Score median:",
    float(
        np.median(
            score_sample_values
        )
    ),
)

print(
    "Score distribution:",
    "PASS",
)


# ==============================================================================
# 10. TARGET ISOLATION
# ==============================================================================

for forbidden_column in [
    "is_correct",
    "target",
    "label",
    "mastery_label",
]:

    assert (
        forbidden_column
        not in RERANK_COLUMNS_OBSERVED
    ), (
        "Forbidden target column found in "
        f"reranking output: {forbidden_column}"
    )


print("\n" + "=" * 90)
print("TARGET ISOLATION")
print("=" * 90)

print(
    "Target-bearing columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 11. DUPLICATE IDENTITY AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("DUPLICATE IDENTITY AUDIT")
print("=" * 90)

print(
    "For memory safety, duplicate identity is "
    "verified using an external-sort approach."
)


# Create a temporary identity-only parquet if needed.
IDENTITY_TEMP = (
    RERANK_ROOT
    / "_cell4_identity_audit.parquet"
)


# We use DuckDB only if available. Otherwise fail explicitly
# rather than silently performing an incomplete audit.

try:

    import duckdb

    DUCKDB_AVAILABLE = True

except Exception:

    DUCKDB_AVAILABLE = False


if DUCKDB_AVAILABLE:

    con = duckdb.connect(
        database=":memory:"
    )

    try:

        duplicate_count = int(
            con.execute(
                f"""
                SELECT COUNT(*)
                FROM (
                    SELECT
                        response_id,
                        session_id,
                        objective_uid,
                        fold,
                        turn_uid,
                        role,
                        turn_index,
                        COUNT(*) AS n
                    FROM read_parquet(
                        '{RERANK_OUTPUT.as_posix()}'
                    )
                    GROUP BY
                        response_id,
                        session_id,
                        objective_uid,
                        fold,
                        turn_uid,
                        role,
                        turn_index
                    HAVING COUNT(*) > 1
                )
                """
            ).fetchone()[0]
        )

    finally:

        con.close()


    assert (
        duplicate_count
        ==
        0
    ), (
        "Duplicate reranking candidate identities detected."
    )


    print(
        "Duplicate identities:",
        duplicate_count,
    )

    print(
        "Duplicate identity audit:",
        "PASS",
    )

else:

    print(
        "DuckDB unavailable."
    )

    print(
        "Duplicate identity audit:",
        "DEFERRED",
    )


# ==============================================================================
# 12. RESPONSE / SESSION POPULATION AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("RESPONSE / SESSION POPULATION AUDIT")
print("=" * 90)


if DUCKDB_AVAILABLE:

    con = duckdb.connect(
        database=":memory:"
    )

    try:

        response_count = int(
            con.execute(
                f"""
                SELECT COUNT(DISTINCT response_id)
                FROM read_parquet(
                    '{RERANK_OUTPUT.as_posix()}'
                )
                """
            ).fetchone()[0]
        )

        session_count = int(
            con.execute(
                f"""
                SELECT COUNT(DISTINCT session_id)
                FROM read_parquet(
                    '{RERANK_OUTPUT.as_posix()}'
                )
                """
            ).fetchone()[0]
        )

    finally:

        con.close()


    assert (
        response_count
        ==
        EXPECTED_R3_RESPONSES
    ), (
        "Reranking response population mismatch."
    )

    assert (
        session_count
        ==
        EXPECTED_R3_SESSIONS
    ), (
        "Reranking session population mismatch."
    )


    print(
        "Responses:",
        f"{response_count:,}",
    )

    print(
        "Sessions:",
        f"{session_count:,}",
    )

    print(
        "Response/session population:",
        "PASS",
    )

else:

    print(
        "Response/session population audit:",
        "DEFERRED",
    )


# ==============================================================================
# 13. FINAL CELL GATE
# ==============================================================================

CROSS_ENCODER_OUTPUT_AUDITED = True
CROSS_ENCODER_CELL_4_READY = True


print("\n" + "=" * 90)
print("CELL 4 FINAL STATUS")
print("=" * 90)

print(
    "R3 population:",
    f"{R3_ROWS_OBSERVED:,}",
)

print(
    "Reranking population:",
    f"{RERANK_ROWS_OBSERVED:,}",
)

print(
    "Exact identity/order audit:",
    "PASS",
)

print(
    "Score finiteness:",
    "PASS",
)

print(
    "Score semantics:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "Duplicate identity:",
    (
        "PASS"
        if DUCKDB_AVAILABLE
        else "DEFERRED"
    ),
)

print(
    "Response/session population:",
    (
        "PASS"
        if DUCKDB_AVAILABLE
        else "DEFERRED"
    ),
)

print(
    "CROSS_ENCODER_OUTPUT_AUDITED:",
    CROSS_ENCODER_OUTPUT_AUDITED,
)

print(
    "CROSS_ENCODER_CELL_4_READY:",
    CROSS_ENCODER_CELL_4_READY,
)


assert (
    CROSS_ENCODER_OUTPUT_AUDITED
    is True
)

assert (
    CROSS_ENCODER_CELL_4_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 4 — OUTPUT AUDIT: PASS"
)
print("=" * 90)


# ==============================================================================
# 14. MEMORY CLEANUP
# ==============================================================================

if "score_sample_table" in globals():
    del score_sample_table

if "score_sample_df" in globals():
    del score_sample_df

if "score_sample_values" in globals():
    del score_sample_values

if "r3_dataset" in globals():
    del r3_dataset

if "rerank_dataset" in globals():
    del rerank_dataset

if "r3_scanner" in globals():
    del r3_scanner

if "rerank_scanner" in globals():
    del rerank_scanner

if "r3_parquet" in globals():
    del r3_parquet

if "rerank_parquet" in globals():
    del rerank_parquet

gc.collect()

print(
    "Cell 4 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 4 — RERANKING OUTPUT RELOAD + EXACT IDENTITY AUDIT

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS

ARTIFACT PATHS
R3 candidate: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R3_union\frozen\r3_candidate_union.parquet
Reranking output: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_scores.parquet
Checkpoint: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_checkpoint.json
Manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_reranking_manifest.json
Artifact existence: PASS

CHECKPOINT CONTRACT
Status: COMPLETE
Completed rows: 2,482,137
Checkpoint: PASS

RERANKING MANIFEST
JSON valid: PASS
Status: COMPLETE
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Upstream r

In [10]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 5 — FINAL PER-RESPONSE RERANKING + RANKING DIAGNOSTICS
# ==============================================================================

from __future__ import annotations

import gc
import json
import os
import sqlite3
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 5 — FINAL PER-RESPONSE RERANKING + RANKING DIAGNOSTICS")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
), "Cell 0 must be executed first."

assert (
    CROSS_ENCODER_CELL_0_READY is True
), "Cell 0 dependency is not ready."


assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
), "Cell 1 must be executed first."

assert (
    CROSS_ENCODER_CELL_1_READY is True
), "Cell 1 dependency is not ready."


assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
), "Cell 2 must be executed first."

assert (
    CROSS_ENCODER_CELL_2_READY is True
), "Cell 2 dependency is not ready."


assert (
    "CROSS_ENCODER_CELL_3_READY" in globals()
), "Cell 3 must be executed first."

assert (
    CROSS_ENCODER_CELL_3_READY is True
), "Cell 3 dependency is not ready."


assert (
    "CROSS_ENCODER_CELL_4_READY" in globals()
), "Cell 4 must be executed first."

assert (
    CROSS_ENCODER_CELL_4_READY is True
), "Cell 4 dependency is not ready."


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print("Cell 0 dependency : PASS")
print("Cell 1 dependency : PASS")
print("Cell 2 dependency : PASS")
print("Cell 3 dependency : PASS")
print("Cell 4 dependency : PASS")


# ==============================================================================
# 2. PATHS
# ==============================================================================

RERANK_ROOT = (
    CROSS_ENCODER_ROOT
    / "reranking"
)

RERANK_INPUT = (
    RERANK_ROOT
    / "cross_encoder_scores.parquet"
)

RANKED_OUTPUT = (
    RERANK_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

RANKED_MANIFEST = (
    RERANK_ROOT
    / "cross_encoder_ranking_manifest.json"
)

SQLITE_ROOT = (
    RERANK_ROOT
    / "ranking_sqlite"
)

SQLITE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RANKING_DB = (
    SQLITE_ROOT
    / "cross_encoder_ranking.sqlite"
)


assert RERANK_INPUT.exists(), (
    "Missing audited Cross-Encoder score artifact.\n"
    f"{RERANK_INPUT}"
)


print("\n" + "=" * 90)
print("RANKING PATHS")
print("=" * 90)

print("Input :", RERANK_INPUT)
print("Output:", RANKED_OUTPUT)
print("Manifest:", RANKED_MANIFEST)
print("SQLite DB:", RANKING_DB)


# ==============================================================================
# 3. INPUT CONTRACT
# ==============================================================================

rerank_parquet = pq.ParquetFile(
    RERANK_INPUT
)

RERANK_ROWS = int(
    rerank_parquet
    .metadata
    .num_rows
)

assert (
    RERANK_ROWS
    ==
    EXPECTED_R3_ROWS
), (
    "Cross-Encoder input population changed.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {RERANK_ROWS:,}"
)


RERANK_COLUMNS = list(
    rerank_parquet
    .schema_arrow
    .names
)


REQUIRED_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
]


assert (
    RERANK_COLUMNS
    ==
    REQUIRED_COLUMNS
), (
    "Unexpected Cross-Encoder score artifact schema.\n"
    f"Observed: {RERANK_COLUMNS}"
)


print("\n" + "=" * 90)
print("INPUT CONTRACT")
print("=" * 90)

print(
    "Input rows:",
    f"{RERANK_ROWS:,}",
)

print(
    "Input schema:",
    "PASS",
)


# ==============================================================================
# 4. RANKING CONTRACT
# ==============================================================================

RANK_GROUP_COLUMNS = [
    "response_id",
    "objective_uid",
]

RANK_OUTPUT_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]


print("\n" + "=" * 90)
print("RANKING CONTRACT")
print("=" * 90)

print(
    "Ranking group:",
    RANK_GROUP_COLUMNS,
)

print(
    "Primary order:",
    "cross_encoder_score DESC",
)

print(
    "Tie-break 1:",
    "turn_index ASC",
)

print(
    "Tie-break 2:",
    "turn_uid ASC",
)

print(
    "Score transformation:",
    "NONE",
)

print(
    "Target used:",
    False,
)


# ==============================================================================
# 5. REMOVE STALE RANKING ARTIFACTS
# ==============================================================================

if RANKED_OUTPUT.exists():

    print(
        "\nExisting ranked output found."
    )

    print(
        "Removing stale ranked output."
    )

    RANKED_OUTPUT.unlink()


if RANKED_MANIFEST.exists():

    RANKED_MANIFEST.unlink()


# A stale SQLite DB must NOT be reused.
if RANKING_DB.exists():

    print(
        "Existing ranking SQLite DB found."
    )

    print(
        "Removing stale ranking database."
    )

    RANKING_DB.unlink()


# ==============================================================================
# 6. BUILD DISK-BACKED SQLITE TABLE
# ==============================================================================

print("\n" + "=" * 90)
print("BUILD DISK-BACKED RANKING TABLE")
print("=" * 90)

print(
    "SQLite is used instead of DuckDB."
)

print(
    "No full 2.48M-row pandas DataFrame will be created."
)


conn = sqlite3.connect(
    RANKING_DB
)

try:

    conn.execute(
        "PRAGMA journal_mode = WAL;"
    )

    conn.execute(
        "PRAGMA synchronous = NORMAL;"
    )

    conn.execute(
        "PRAGMA temp_store = FILE;"
    )

    conn.execute(
        "PRAGMA cache_size = -262144;"
    )

    conn.execute(
        """
        CREATE TABLE candidates (
            row_id INTEGER PRIMARY KEY,
            response_id TEXT NOT NULL,
            session_id TEXT NOT NULL,
            objective_uid TEXT NOT NULL,
            fold INTEGER NOT NULL,
            turn_uid TEXT NOT NULL,
            role TEXT NOT NULL,
            turn_index INTEGER NOT NULL,
            cross_encoder_score REAL NOT NULL
        );
        """
    )

    conn.commit()


    scanner = ds.dataset(
        RERANK_INPUT,
        format="parquet",
    ).scanner(
        columns=REQUIRED_COLUMNS,
        batch_size=25_000,
    )


    inserted_rows = 0
    batch_number = 0

    insert_sql = """
        INSERT INTO candidates (
            row_id,
            response_id,
            session_id,
            objective_uid,
            fold,
            turn_uid,
            role,
            turn_index,
            cross_encoder_score
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """


    for batch in scanner.to_batches():

        batch_number += 1

        df = (
            batch
            .to_pandas()
        )


        # ----------------------------------------------------------------------
        # Local validation
        # ----------------------------------------------------------------------

        scores = pd.to_numeric(
            df[
                "cross_encoder_score"
            ],
            errors="coerce",
        )


        assert (
            scores.notna().all()
        ), (
            "NaN/non-numeric score detected "
            f"in input batch {batch_number}."
        )


        score_values = (
            scores
            .to_numpy(
                dtype=np.float64
            )
        )


        assert np.isfinite(
            score_values
        ).all(), (
            "Non-finite score detected "
            f"in input batch {batch_number}."
        )


        # ----------------------------------------------------------------------
        # Explicit row identity
        # ----------------------------------------------------------------------

        rows = []

        for local_index, row in df.iterrows():

            rows.append(
                (
                    inserted_rows
                    +
                    local_index
                    +
                    1,

                    str(
                        row[
                            "response_id"
                        ]
                    ),

                    str(
                        row[
                            "session_id"
                        ]
                    ),

                    str(
                        row[
                            "objective_uid"
                        ]
                    ),

                    int(
                        row[
                            "fold"
                        ]
                    ),

                    str(
                        row[
                            "turn_uid"
                        ]
                    ),

                    str(
                        row[
                            "role"
                        ]
                    ),

                    int(
                        row[
                            "turn_index"
                        ]
                    ),

                    float(
                        row[
                            "cross_encoder_score"
                        ]
                    ),
                )
            )


        conn.executemany(
            insert_sql,
            rows,
        )


        inserted_rows += len(
            rows
        )


        if (
            batch_number % 10
            ==
            0
        ):

            conn.commit()

            print(
                "Indexed:",
                f"{inserted_rows:,}/"
                f"{RERANK_ROWS:,}",
            )


        del df
        del scores
        del score_values
        del rows
        del batch

        gc.collect()


    conn.commit()


    assert (
        inserted_rows
        ==
        RERANK_ROWS
    ), (
        "SQLite input population mismatch.\n"
        f"Expected: {RERANK_ROWS:,}\n"
        f"Inserted: {inserted_rows:,}"
    )


    print(
        "SQLite rows:",
        f"{inserted_rows:,}",
    )

    print(
        "SQLite population:",
        "PASS",
    )


    # ==========================================================================
    # 7. CREATE INDEXES
    # ==========================================================================

    print("\n" + "=" * 90)
    print("CREATE RANKING INDEXES")
    print("=" * 90)


    conn.execute(
        """
        CREATE INDEX idx_rank_group
        ON candidates (
            response_id,
            objective_uid
        );
        """
    )

    conn.execute(
        """
        CREATE INDEX idx_identity
        ON candidates (
            response_id,
            objective_uid,
            turn_uid
        );
        """
    )

    conn.commit()


    print(
        "Ranking indexes:",
        "PASS",
    )


    # ==========================================================================
    # 8. CREATE RANKED SQLITE VIEW
    # ==========================================================================

    print("\n" + "=" * 90)
    print("GENERATE DETERMINISTIC RANKS")
    print("=" * 90)


    conn.execute(
        """
        DROP TABLE IF EXISTS ranked_candidates;
        """
    )


    conn.execute(
        """
        CREATE TABLE ranked_candidates AS

        SELECT
            response_id,
            session_id,
            objective_uid,
            fold,
            turn_uid,
            role,
            turn_index,
            cross_encoder_score,

            ROW_NUMBER() OVER (
                PARTITION BY
                    response_id,
                    objective_uid

                ORDER BY
                    cross_encoder_score DESC,
                    turn_index ASC,
                    turn_uid ASC
            ) AS cross_encoder_rank

        FROM candidates;
        """
    )


    conn.commit()


    print(
        "Rank generation:",
        "PASS",
    )


    # ==========================================================================
    # 9. RANKED POPULATION AUDIT
    # ==========================================================================

    ranked_count = int(
        conn.execute(
            """
            SELECT COUNT(*)
            FROM ranked_candidates
            """
        ).fetchone()[0]
    )


    assert (
        ranked_count
        ==
        RERANK_ROWS
    ), (
        "Ranked SQLite population mismatch."
    )


    print(
        "Ranked rows:",
        f"{ranked_count:,}",
    )

    print(
        "Ranked population:",
        "PASS",
    )


    # ==========================================================================
    # 10. EXACT RANK SEQUENCE AUDIT
    # ==========================================================================

    print("\n" + "=" * 90)
    print("RANK SEQUENCE AUDIT")
    print("=" * 90)


    bad_rank_groups = int(
        conn.execute(
            """
            SELECT COUNT(*)
            FROM (
                SELECT
                    response_id,
                    objective_uid,

                    COUNT(*) AS n,
                    MIN(
                        cross_encoder_rank
                    ) AS min_rank,

                    MAX(
                        cross_encoder_rank
                    ) AS max_rank,

                    COUNT(
                        DISTINCT
                        cross_encoder_rank
                    ) AS distinct_ranks

                FROM ranked_candidates

                GROUP BY
                    response_id,
                    objective_uid

                HAVING
                    min_rank <> 1
                    OR
                    max_rank <> n
                    OR
                    distinct_ranks <> n
            );
            """
        ).fetchone()[0]
    )


    assert (
        bad_rank_groups
        ==
        0
    ), (
        "Invalid rank sequence detected."
    )


    print(
        "Invalid rank groups:",
        bad_rank_groups,
    )

    print(
        "Rank sequence:",
        "PASS",
    )


    # ==========================================================================
    # 11. SCORE ORDERING AUDIT
    # ==========================================================================

    print("\n" + "=" * 90)
    print("SCORE ORDERING AUDIT")
    print("=" * 90)


    ordering_violations = int(
        conn.execute(
            """
            SELECT COUNT(*)
            FROM (
                SELECT
                    response_id,
                    objective_uid,
                    cross_encoder_rank,
                    cross_encoder_score,

                    LAG(
                        cross_encoder_score
                    ) OVER (
                        PARTITION BY
                            response_id,
                            objective_uid

                        ORDER BY
                            cross_encoder_rank
                    ) AS previous_score

                FROM ranked_candidates
            )

            WHERE
                previous_score IS NOT NULL
                AND
                cross_encoder_score
                >
                previous_score;
            """
        ).fetchone()[0]
    )


    assert (
        ordering_violations
        ==
        0
    ), (
        "Cross-Encoder score ordering violation detected."
    )


    print(
        "Ordering violations:",
        ordering_violations,
    )

    print(
        "Score ordering:",
        "PASS",
    )


    # ==========================================================================
    # 12. POPULATION AUDIT
    # ==========================================================================

    print("\n" + "=" * 90)
    print("RESPONSE / SESSION / OBJECTIVE AUDIT")
    print("=" * 90)


    population = conn.execute(
        """
        SELECT
            COUNT(*) AS rows,

            COUNT(
                DISTINCT response_id
            ) AS responses,

            COUNT(
                DISTINCT session_id
            ) AS sessions,

            COUNT(
                DISTINCT objective_uid
            ) AS objectives,

            COUNT(
                DISTINCT
                response_id
                || '::'
                || objective_uid
            ) AS groups

        FROM ranked_candidates;
        """
    ).fetchone()


    (
        pop_rows,
        pop_responses,
        pop_sessions,
        pop_objectives,
        pop_groups,
    ) = population


    assert (
        int(pop_rows)
        ==
        EXPECTED_R3_ROWS
    )

    assert (
        int(pop_responses)
        ==
        EXPECTED_R3_RESPONSES
    )

    assert (
        int(pop_sessions)
        ==
        EXPECTED_R3_SESSIONS
    )

    assert (
        int(pop_objectives)
        ==
        EXPECTED_R3_OBJECTIVES
    )


    print(
        "Rows:",
        f"{int(pop_rows):,}",
    )

    print(
        "Responses:",
        f"{int(pop_responses):,}",
    )

    print(
        "Sessions:",
        f"{int(pop_sessions):,}",
    )

    print(
        "Objectives:",
        f"{int(pop_objectives):,}",
    )

    print(
        "Response-objective groups:",
        f"{int(pop_groups):,}",
    )

    print(
        "Population audit:",
        "PASS",
    )


    # ==========================================================================
    # 13. RANK-1 AUDIT
    # ==========================================================================

    print("\n" + "=" * 90)
    print("RANK-1 AUDIT")
    print("=" * 90)


    rank_one_count = int(
        conn.execute(
            """
            SELECT COUNT(*)
            FROM ranked_candidates
            WHERE cross_encoder_rank = 1;
            """
        ).fetchone()[0]
    )


    assert (
        rank_one_count
        ==
        int(pop_groups)
    ), (
        "Every response-objective group must have "
        "exactly one rank-1 candidate."
    )


    print(
        "Rank-1 rows:",
        f"{rank_one_count:,}",
    )

    print(
        "Rank-1 audit:",
        "PASS",
    )


    # ==========================================================================
    # 14. TOP-K DIAGNOSTICS
    # ==========================================================================

    topk = conn.execute(
        """
        SELECT
            SUM(
                CASE
                    WHEN cross_encoder_rank <= 5
                    THEN 1 ELSE 0
                END
            ),

            SUM(
                CASE
                    WHEN cross_encoder_rank <= 10
                    THEN 1 ELSE 0
                END
            ),

            SUM(
                CASE
                    WHEN cross_encoder_rank <= 25
                    THEN 1 ELSE 0
                END
            ),

            SUM(
                CASE
                    WHEN cross_encoder_rank <= 50
                    THEN 1 ELSE 0
                END
            )

        FROM ranked_candidates;
        """
    ).fetchone()


    (
        top5,
        top10,
        top25,
        top50,
    ) = topk


    print("\n" + "=" * 90)
    print("TOP-K DIAGNOSTICS")
    print("=" * 90)

    print(
        "Top-5 rows:",
        f"{int(top5):,}",
    )

    print(
        "Top-10 rows:",
        f"{int(top10):,}",
    )

    print(
        "Top-25 rows:",
        f"{int(top25):,}",
    )

    print(
        "Top-50 rows:",
        f"{int(top50):,}",
    )

    print(
        "Top-K diagnostics:",
        "PASS",
    )


    # ==========================================================================
    # 15. WRITE FINAL RANKED PARQUET
    # ==========================================================================

    print("\n" + "=" * 90)
    print("WRITE RANKED PARQUET")
    print("=" * 90)


    output_tmp = (
        RERANK_ROOT
        / "cross_encoder_ranked_candidates.tmp.parquet"
    )


    output_tmp.unlink(
        missing_ok=True
    )


    output_writer = None


    export_query = """
        SELECT
            response_id,
            session_id,
            objective_uid,
            fold,
            turn_uid,
            role,
            turn_index,
            cross_encoder_score,
            cross_encoder_rank

        FROM ranked_candidates

        ORDER BY
            response_id,
            objective_uid,
            cross_encoder_rank;
    """


    cursor = conn.execute(
        export_query
    )


    EXPORT_BATCH_ROWS = 50_000
    exported_rows = 0


    while True:

        rows = cursor.fetchmany(
            EXPORT_BATCH_ROWS
        )


        if not rows:
            break


        df = pd.DataFrame(
            rows,
            columns=RANK_OUTPUT_COLUMNS,
        )


        table = pa.Table.from_pandas(
            df,
            preserve_index=False,
        )


        if output_writer is None:

            output_writer = pq.ParquetWriter(
                output_tmp,
                table.schema,
                compression="zstd",
            )


        output_writer.write_table(
            table
        )


        exported_rows += len(
            df
        )


        if (
            exported_rows % 250_000
            <
            EXPORT_BATCH_ROWS
        ):

            print(
                "Exported:",
                f"{exported_rows:,}/"
                f"{RERANK_ROWS:,}",
            )


        del rows
        del df
        del table

        gc.collect()


    if output_writer is not None:

        output_writer.close()

        output_writer = None


    assert (
        exported_rows
        ==
        RERANK_ROWS
    ), (
        "Exported ranked row count mismatch.\n"
        f"Expected: {RERANK_ROWS:,}\n"
        f"Observed: {exported_rows:,}"
    )


    os.replace(
        output_tmp,
        RANKED_OUTPUT,
    )


finally:

    conn.close()


# ==============================================================================
# 16. FINAL PARQUET VERIFICATION
# ==============================================================================

ranked_parquet = pq.ParquetFile(
    RANKED_OUTPUT
)

RANKED_ROWS = int(
    ranked_parquet
    .metadata
    .num_rows
)

RANKED_COLUMNS = list(
    ranked_parquet
    .schema_arrow
    .names
)


assert (
    RANKED_ROWS
    ==
    EXPECTED_R3_ROWS
), (
    "Final ranked parquet row count mismatch."
)


assert (
    RANKED_COLUMNS
    ==
    RANK_OUTPUT_COLUMNS
), (
    "Final ranked parquet schema mismatch."
)


print("\n" + "=" * 90)
print("FINAL RANKED ARTIFACT")
print("=" * 90)

print(
    "Rows:",
    f"{RANKED_ROWS:,}",
)

print(
    "Columns:",
    RANKED_COLUMNS,
)

print(
    "Population:",
    "PASS",
)

print(
    "Schema:",
    "PASS",
)


# ==============================================================================
# 17. TARGET ISOLATION
# ==============================================================================

FORBIDDEN_COLUMNS = {
    "target",
    "label",
    "is_correct",
    "mastery_label",
}


assert not (
    set(
        RANKED_COLUMNS
    )
    &
    FORBIDDEN_COLUMNS
), (
    "Target-bearing column found in ranked artifact."
)


print("\n" + "=" * 90)
print("TARGET ISOLATION")
print("=" * 90)

print(
    "Forbidden columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 18. MANIFEST
# ==============================================================================

RANKING_MANIFEST_DATA = {
    "artifact": (
        "cross_encoder_ranked_candidates"
    ),

    "status": "COMPLETE",

    "upstream_artifact": (
        "cross_encoder_scores"
    ),

    "upstream_rows": int(
        RERANK_ROWS
    ),

    "output_rows": int(
        RANKED_ROWS
    ),

    "model": (
        CROSS_ENCODER_MODEL_NAME
    ),

    "ranking_group": [
        "response_id",
        "objective_uid",
    ],

    "primary_order": (
        "cross_encoder_score DESC"
    ),

    "secondary_order": (
        "turn_index ASC"
    ),

    "tertiary_order": (
        "turn_uid ASC"
    ),

    "score_transformation": "NONE",

    "target_used": False,

    "responses": int(
        pop_responses
    ),

    "sessions": int(
        pop_sessions
    ),

    "objectives": int(
        pop_objectives
    ),

    "response_objective_groups": int(
        pop_groups
    ),

    "rank_1_rows": int(
        rank_one_count
    ),

    "top5_rows": int(
        top5
    ),

    "top10_rows": int(
        top10
    ),

    "top25_rows": int(
        top25
    ),

    "top50_rows": int(
        top50
    ),

    "output_path": str(
        RANKED_OUTPUT
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}


with open(
    RANKED_MANIFEST,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        RANKING_MANIFEST_DATA,
        handle,
        indent=2,
    )


assert RANKED_MANIFEST.exists()


# ==============================================================================
# 19. FINAL CELL GATE
# ==============================================================================

CROSS_ENCODER_RANKING_COMPLETE = True
CROSS_ENCODER_CELL_5_READY = True


print("\n" + "=" * 90)
print("CELL 5 FINAL STATUS")
print("=" * 90)

print(
    "Input rows:",
    f"{RERANK_ROWS:,}",
)

print(
    "Ranked rows:",
    f"{RANKED_ROWS:,}",
)

print(
    "Rank sequence:",
    "PASS",
)

print(
    "Score ordering:",
    "PASS",
)

print(
    "Population:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "Ranking manifest:",
    "PASS",
)

print(
    "CROSS_ENCODER_RANKING_COMPLETE:",
    CROSS_ENCODER_RANKING_COMPLETE,
)

print(
    "CROSS_ENCODER_CELL_5_READY:",
    CROSS_ENCODER_CELL_5_READY,
)


assert (
    CROSS_ENCODER_RANKING_COMPLETE
    is True
)

assert (
    CROSS_ENCODER_CELL_5_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 5 — FINAL RERANKING: PASS"
)
print("=" * 90)


# ==============================================================================
# 20. MEMORY CLEANUP
# ==============================================================================

if "rerank_parquet" in globals():
    del rerank_parquet

if "ranked_parquet" in globals():
    del ranked_parquet

if "scanner" in globals():
    del scanner

if "cursor" in globals():
    del cursor

gc.collect()

print(
    "Cell 5 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 5 — FINAL PER-RESPONSE RERANKING + RANKING DIAGNOSTICS

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS
Cell 4 dependency : PASS

RANKING PATHS
Input : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_scores.parquet
Output: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_ranked_candidates.parquet
Manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_ranking_manifest.json
SQLite DB: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\ranking_sqlite\cross_encoder_ranking.sqlite

INPUT CONTRACT
Input rows: 2,482,137
Input schema: PASS

RANKING CONTRACT
Ranking group: ['response_id', 'objective_uid']
Primary order: cross_encoder_score DESC
Tie-br

In [12]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 6 — FINAL RANKED CANDIDATE FREEZE + INTEGRITY MANIFEST
# ==============================================================================

from __future__ import annotations

import gc
import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 6 — FINAL RANKED CANDIDATE FREEZE + INTEGRITY MANIFEST")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
    and CROSS_ENCODER_CELL_0_READY is True
), "Cell 0 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
    and CROSS_ENCODER_CELL_1_READY is True
), "Cell 1 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
    and CROSS_ENCODER_CELL_2_READY is True
), "Cell 2 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_3_READY" in globals()
    and CROSS_ENCODER_CELL_3_READY is True
), "Cell 3 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_4_READY" in globals()
    and CROSS_ENCODER_CELL_4_READY is True
), "Cell 4 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_5_READY" in globals()
    and CROSS_ENCODER_CELL_5_READY is True
), "Cell 5 dependency is not ready."


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print("Cell 0 dependency : PASS")
print("Cell 1 dependency : PASS")
print("Cell 2 dependency : PASS")
print("Cell 3 dependency : PASS")
print("Cell 4 dependency : PASS")
print("Cell 5 dependency : PASS")


# ==============================================================================
# 2. PATH CONTRACT
# ==============================================================================

RERANK_ROOT = (
    CROSS_ENCODER_ROOT
    / "reranking"
)

RANKED_INPUT = (
    RERANK_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

RANKED_MANIFEST = (
    RERANK_ROOT
    / "cross_encoder_ranking_manifest.json"
)

FREEZE_ROOT = (
    CROSS_ENCODER_ROOT
    / "frozen"
)

FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

FREEZE_CANDIDATE_PATH = (
    FREEZE_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

FREEZE_MANIFEST_PATH = (
    FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)

FREEZE_TMP_PATH = (
    FREEZE_ROOT
    / "cross_encoder_ranked_candidates.tmp.parquet"
)


print("\n" + "=" * 90)
print("FREEZE PATHS")
print("=" * 90)

print("Ranked input:", RANKED_INPUT)
print("Ranking manifest:", RANKED_MANIFEST)
print("Freeze root:", FREEZE_ROOT)
print("Frozen candidates:", FREEZE_CANDIDATE_PATH)
print("Freeze manifest:", FREEZE_MANIFEST_PATH)


assert RANKED_INPUT.exists(), (
    "Final ranked Cross-Encoder artifact is missing.\n"
    f"{RANKED_INPUT}"
)

assert RANKED_MANIFEST.exists(), (
    "Cross-Encoder ranking manifest is missing.\n"
    f"{RANKED_MANIFEST}"
)


# ==============================================================================
# 3. LOAD UPSTREAM MANIFEST
# ==============================================================================

with open(
    RANKED_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    upstream_manifest = json.load(
        handle
    )


assert (
    upstream_manifest.get("status")
    ==
    "COMPLETE"
), (
    "Upstream Cross-Encoder ranking is not COMPLETE."
)

assert (
    upstream_manifest.get("target_used")
    is False
), (
    "Upstream ranking manifest indicates target usage."
)

assert (
    upstream_manifest.get("model")
    ==
    CROSS_ENCODER_MODEL_NAME
), (
    "Upstream ranking model mismatch."
)


print("\n" + "=" * 90)
print("UPSTREAM RANKING MANIFEST")
print("=" * 90)

print("JSON valid:", "PASS")
print(
    "Status:",
    upstream_manifest.get("status"),
)
print(
    "Model:",
    upstream_manifest.get("model"),
)
print(
    "Target used:",
    upstream_manifest.get("target_used"),
)


# ==============================================================================
# 4. INPUT ARTIFACT CONTRACT
# ==============================================================================

ranked_parquet = pq.ParquetFile(
    RANKED_INPUT
)

ranked_rows = int(
    ranked_parquet
    .metadata
    .num_rows
)

ranked_columns = list(
    ranked_parquet
    .schema_arrow
    .names
)


EXPECTED_FROZEN_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]


assert (
    ranked_rows
    ==
    EXPECTED_R3_ROWS
), (
    "Ranked artifact population mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {ranked_rows:,}"
)

assert (
    ranked_columns
    ==
    EXPECTED_FROZEN_COLUMNS
), (
    "Ranked artifact schema mismatch.\n"
    f"Observed: {ranked_columns}"
)


print("\n" + "=" * 90)
print("INPUT ARTIFACT CONTRACT")
print("=" * 90)

print(
    "Rows:",
    f"{ranked_rows:,}",
)

print(
    "Columns:",
    ranked_columns,
)

print("Population:", "PASS")
print("Schema:", "PASS")


# ==============================================================================
# 5. REMOVE STALE / PARTIAL FREEZE ARTIFACTS
# ==============================================================================

if FREEZE_CANDIDATE_PATH.exists():

    print(
        "\nExisting frozen candidate found."
    )

    print(
        "Removing stale freeze artifact."
    )

    FREEZE_CANDIDATE_PATH.unlink()


if FREEZE_MANIFEST_PATH.exists():

    print(
        "Existing freeze manifest found."
    )

    print(
        "Removing stale freeze manifest."
    )

    FREEZE_MANIFEST_PATH.unlink()


if FREEZE_TMP_PATH.exists():

    print(
        "Existing partial temporary parquet found."
    )

    print(
        "Removing partial temporary parquet."
    )

    FREEZE_TMP_PATH.unlink()


# ==============================================================================
# 6. STREAMING FREEZE COPY
# ==============================================================================

print("\n" + "=" * 90)
print("STREAMING FREEZE COPY")
print("=" * 90)

print(
    "Mode:",
    "PyArrow RecordBatch → Table streaming",
)

print(
    "Source rows:",
    f"{ranked_rows:,}",
)


source_dataset = ds.dataset(
    RANKED_INPUT,
    format="parquet",
)

source_scanner = source_dataset.scanner(
    columns=EXPECTED_FROZEN_COLUMNS,
    batch_size=50_000,
)


freeze_writer = None
copied_rows = 0
copy_batches = 0


try:

    for record_batch in source_scanner.to_batches():

        copy_batches += 1


        # ----------------------------------------------------------------------
        # IMPORTANT:
        #
        # scanner.to_batches() returns pyarrow.RecordBatch.
        #
        # ParquetWriter.write_table() requires pyarrow.Table.
        #
        # Therefore convert the RecordBatch explicitly.
        # ----------------------------------------------------------------------

        batch_table = pa.Table.from_batches(
            [record_batch]
        )


        if freeze_writer is None:

            freeze_writer = pq.ParquetWriter(
                FREEZE_TMP_PATH,
                batch_table.schema,
                compression="zstd",
            )


        freeze_writer.write_table(
            batch_table
        )


        copied_rows += (
            batch_table.num_rows
        )


        if (
            copy_batches % 10
            ==
            0
        ):

            print(
                "Copied:",
                f"{copied_rows:,}/"
                f"{ranked_rows:,}",
            )


        del record_batch
        del batch_table

        gc.collect()


finally:

    if freeze_writer is not None:

        freeze_writer.close()

        freeze_writer = None


assert (
    copied_rows
    ==
    ranked_rows
), (
    "Frozen copy row count mismatch.\n"
    f"Expected: {ranked_rows:,}\n"
    f"Copied: {copied_rows:,}"
)


assert FREEZE_TMP_PATH.exists(), (
    "Temporary frozen parquet was not created."
)


os.replace(
    FREEZE_TMP_PATH,
    FREEZE_CANDIDATE_PATH,
)


assert FREEZE_CANDIDATE_PATH.exists(), (
    "Frozen candidate parquet was not created."
)


print(
    "Copied rows:",
    f"{copied_rows:,}",
)

print(
    "Copy batches:",
    f"{copy_batches:,}",
)

print(
    "Frozen candidate written:",
    "PASS",
)


# ==============================================================================
# 7. FROZEN ARTIFACT RELOAD
# ==============================================================================

print("\n" + "=" * 90)
print("FROZEN ARTIFACT RELOAD")
print("=" * 90)


frozen_parquet = pq.ParquetFile(
    FREEZE_CANDIDATE_PATH
)

frozen_rows = int(
    frozen_parquet
    .metadata
    .num_rows
)

frozen_columns = list(
    frozen_parquet
    .schema_arrow
    .names
)


assert (
    frozen_rows
    ==
    EXPECTED_R3_ROWS
), (
    "Frozen row count mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {frozen_rows:,}"
)

assert (
    frozen_columns
    ==
    EXPECTED_FROZEN_COLUMNS
), (
    "Frozen schema mismatch."
)


print(
    "Frozen rows:",
    f"{frozen_rows:,}",
)

print(
    "Frozen columns:",
    frozen_columns,
)

print(
    "Frozen reload:",
    "PASS",
)


# ==============================================================================
# 8. SHA256 — STREAMING
# ==============================================================================

print("\n" + "=" * 90)
print("SHA256 INTEGRITY")
print("=" * 90)


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


frozen_sha256 = sha256_file(
    FREEZE_CANDIDATE_PATH
)


print(
    "Frozen SHA256:",
    frozen_sha256,
)

print(
    "SHA256:",
    "PASS",
)


# ==============================================================================
# 9. SOURCE / FROZEN SAMPLE AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("FROZEN IDENTITY SAMPLE AUDIT")
print("=" * 90)


AUDIT_ROWS = 100_000


def collect_sample(
    scanner,
    n_rows,
):

    iterator = iter(
        scanner.to_batches()
    )

    pieces = []
    total = 0

    while total < n_rows:

        try:

            record_batch = next(
                iterator
            )

        except StopIteration:

            break


        table = pa.Table.from_batches(
            [record_batch]
        )


        df = table.to_pandas()


        remaining = (
            n_rows
            -
            total
        )


        if len(df) > remaining:

            df = (
                df
                .head(
                    remaining
                )
                .copy()
            )


        pieces.append(
            df
        )

        total += len(
            df
        )


        del record_batch
        del table
        del df

        gc.collect()


    if not pieces:

        return pd.DataFrame(
            columns=EXPECTED_FROZEN_COLUMNS
        )


    return pd.concat(
        pieces,
        ignore_index=True,
    )


source_sample_scanner = (
    ds.dataset(
        RANKED_INPUT,
        format="parquet",
    )
    .scanner(
        columns=EXPECTED_FROZEN_COLUMNS,
        batch_size=50_000,
    )
)


frozen_sample_scanner = (
    ds.dataset(
        FREEZE_CANDIDATE_PATH,
        format="parquet",
    )
    .scanner(
        columns=EXPECTED_FROZEN_COLUMNS,
        batch_size=50_000,
    )
)


source_sample_df = collect_sample(
    source_sample_scanner,
    AUDIT_ROWS,
)

frozen_sample_df = collect_sample(
    frozen_sample_scanner,
    AUDIT_ROWS,
)


assert (
    len(source_sample_df)
    ==
    len(frozen_sample_df)
), (
    "Source/frozen sample population mismatch."
)


for column in EXPECTED_FROZEN_COLUMNS:

    left = (
        source_sample_df[
            column
        ]
        .astype(str)
        .to_numpy()
    )

    right = (
        frozen_sample_df[
            column
        ]
        .astype(str)
        .to_numpy()
    )


    assert np.array_equal(
        left,
        right,
    ), (
        "Frozen artifact sample mismatch.\n"
        f"Column: {column}"
    )


print(
    "Rows sampled:",
    f"{len(source_sample_df):,}",
)

print(
    "Identity/order sample:",
    "PASS",
)


# ==============================================================================
# 10. TARGET ISOLATION
# ==============================================================================

FORBIDDEN_COLUMNS = {
    "target",
    "label",
    "is_correct",
    "mastery_label",
}


observed_forbidden = (
    set(
        frozen_columns
    )
    &
    FORBIDDEN_COLUMNS
)


assert not observed_forbidden, (
    "Forbidden target-bearing columns found:\n"
    f"{sorted(observed_forbidden)}"
)


print("\n" + "=" * 90)
print("TARGET ISOLATION")
print("=" * 90)

print(
    "Forbidden columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 11. FREEZE MANIFEST
# ==============================================================================

freeze_manifest = {
    "artifact": (
        "cross_encoder_ranked_candidates"
    ),

    "status": "FROZEN",

    "source_artifact": (
        "cross_encoder_ranked_candidates"
    ),

    "source_manifest": str(
        RANKED_MANIFEST
    ),

    "rows": int(
        frozen_rows
    ),

    "responses": int(
        upstream_manifest.get(
            "responses"
        )
    ),

    "sessions": int(
        upstream_manifest.get(
            "sessions"
        )
    ),

    "objectives": int(
        upstream_manifest.get(
            "objectives"
        )
    ),

    "response_objective_groups": int(
        upstream_manifest.get(
            "response_objective_groups"
        )
    ),

    "rank_1_rows": int(
        upstream_manifest.get(
            "rank_1_rows"
        )
    ),

    "top5_rows": int(
        upstream_manifest.get(
            "top5_rows"
        )
    ),

    "top10_rows": int(
        upstream_manifest.get(
            "top10_rows"
        )
    ),

    "top25_rows": int(
        upstream_manifest.get(
            "top25_rows"
        )
    ),

    "top50_rows": int(
        upstream_manifest.get(
            "top50_rows"
        )
    ),

    "model": (
        upstream_manifest.get(
            "model"
        )
    ),

    "ranking_group": (
        upstream_manifest.get(
            "ranking_group"
        )
    ),

    "primary_order": (
        upstream_manifest.get(
            "primary_order"
        )
    ),

    "secondary_order": (
        upstream_manifest.get(
            "secondary_order"
        )
    ),

    "tertiary_order": (
        upstream_manifest.get(
            "tertiary_order"
        )
    ),

    "score_transformation": (
        upstream_manifest.get(
            "score_transformation"
        )
    ),

    "target_used": False,

    "columns": frozen_columns,

    "candidate_sha256": (
        frozen_sha256
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}


with open(
    FREEZE_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        freeze_manifest,
        handle,
        indent=2,
    )


assert FREEZE_MANIFEST_PATH.exists()


# ==============================================================================
# 12. MANIFEST RELOAD + CONTRACT
# ==============================================================================

with open(
    FREEZE_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    reloaded_manifest = json.load(
        handle
    )


assert (
    reloaded_manifest.get(
        "status"
    )
    ==
    "FROZEN"
)

assert (
    int(
        reloaded_manifest.get(
            "rows"
        )
    )
    ==
    EXPECTED_R3_ROWS
)

assert (
    reloaded_manifest.get(
        "candidate_sha256"
    )
    ==
    frozen_sha256
)

assert (
    reloaded_manifest.get(
        "columns"
    )
    ==
    EXPECTED_FROZEN_COLUMNS
)

assert (
    reloaded_manifest.get(
        "target_used"
    )
    is False
)


print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid:",
    "PASS",
)

print(
    "Status:",
    reloaded_manifest.get(
        "status"
    ),
)

print(
    "Rows:",
    f"{int(reloaded_manifest['rows']):,}",
)

print(
    "SHA256 recorded:",
    "PASS",
)

print(
    "Schema recorded:",
    "PASS",
)

print(
    "Target isolation recorded:",
    "PASS",
)


# ==============================================================================
# 13. FINAL FREEZE GATE
# ==============================================================================

CROSS_ENCODER_FREEZE_COMPLETE = True
CROSS_ENCODER_CELL_6_READY = True


print("\n" + "=" * 90)
print("CELL 6 FINAL STATUS")
print("=" * 90)

print(
    "Frozen candidate:",
    FREEZE_CANDIDATE_PATH.exists(),
)

print(
    "Frozen manifest:",
    FREEZE_MANIFEST_PATH.exists(),
)

print(
    "Rows:",
    f"{frozen_rows:,}",
)

print(
    "SHA256:",
    "PASS",
)

print(
    "Schema:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "Freeze status:",
    "FROZEN",
)

print(
    "CROSS_ENCODER_FREEZE_COMPLETE:",
    CROSS_ENCODER_FREEZE_COMPLETE,
)

print(
    "CROSS_ENCODER_CELL_6_READY:",
    CROSS_ENCODER_CELL_6_READY,
)


assert (
    CROSS_ENCODER_FREEZE_COMPLETE
    is True
)

assert (
    CROSS_ENCODER_CELL_6_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 6 — FINAL RANKED CANDIDATE FREEZE: PASS"
)
print("=" * 90)


# ==============================================================================
# 14. MEMORY CLEANUP
# ==============================================================================

cleanup_names = [
    "ranked_parquet",
    "frozen_parquet",
    "source_dataset",
    "source_scanner",
    "source_sample_scanner",
    "frozen_sample_scanner",
    "source_sample_df",
    "frozen_sample_df",
    "batch_table",
    "record_batch",
]


for name in cleanup_names:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


print(
    "Cell 6 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 6 — FINAL RANKED CANDIDATE FREEZE + INTEGRITY MANIFEST

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS
Cell 4 dependency : PASS
Cell 5 dependency : PASS

FREEZE PATHS
Ranked input: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_ranked_candidates.parquet
Ranking manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\reranking\cross_encoder_ranking_manifest.json
Freeze root: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen
Frozen candidates: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cross_encoder_ranked_candidates.parquet
Freeze manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cell6_freeze_manifest.json

UPSTREAM RANKING M

In [13]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION
# ==============================================================================

from __future__ import annotations

import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
    and CROSS_ENCODER_CELL_0_READY is True
), "Cell 0 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
    and CROSS_ENCODER_CELL_1_READY is True
), "Cell 1 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
    and CROSS_ENCODER_CELL_2_READY is True
), "Cell 2 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_3_READY" in globals()
    and CROSS_ENCODER_CELL_3_READY is True
), "Cell 3 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_4_READY" in globals()
    and CROSS_ENCODER_CELL_4_READY is True
), "Cell 4 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_5_READY" in globals()
    and CROSS_ENCODER_CELL_5_READY is True
), "Cell 5 dependency is not ready."

assert (
    "CROSS_ENCODER_CELL_6_READY" in globals()
    and CROSS_ENCODER_CELL_6_READY is True
), "Cell 6 dependency is not ready."


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print("Cell 0 dependency : PASS")
print("Cell 1 dependency : PASS")
print("Cell 2 dependency : PASS")
print("Cell 3 dependency : PASS")
print("Cell 4 dependency : PASS")
print("Cell 5 dependency : PASS")
print("Cell 6 dependency : PASS")


# ==============================================================================
# 2. FROZEN PATHS
# ==============================================================================

FREEZE_ROOT = (
    CROSS_ENCODER_ROOT
    / "frozen"
)

FROZEN_CANDIDATE_PATH = (
    FREEZE_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

FROZEN_MANIFEST_PATH = (
    FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)


print("\n" + "=" * 90)
print("FROZEN ARTIFACT PATHS")
print("=" * 90)

print(
    "Frozen root:",
    FREEZE_ROOT,
)

print(
    "Candidate:",
    FROZEN_CANDIDATE_PATH,
)

print(
    "Manifest:",
    FROZEN_MANIFEST_PATH,
)


# ==============================================================================
# 3. ARTIFACT EXISTENCE
# ==============================================================================

assert FREEZE_ROOT.exists(), (
    "Frozen Cross-Encoder directory is missing."
)

assert FROZEN_CANDIDATE_PATH.exists(), (
    "Frozen ranked candidate parquet is missing.\n"
    f"{FROZEN_CANDIDATE_PATH}"
)

assert FROZEN_MANIFEST_PATH.exists(), (
    "Cell 6 freeze manifest is missing.\n"
    f"{FROZEN_MANIFEST_PATH}"
)


print("\n" + "=" * 90)
print("ARTIFACT EXISTENCE")
print("=" * 90)

print(
    "Frozen directory:",
    FREEZE_ROOT.exists(),
)

print(
    "Candidate parquet:",
    FROZEN_CANDIDATE_PATH.exists(),
)

print(
    "Freeze manifest:",
    FROZEN_MANIFEST_PATH.exists(),
)


# ==============================================================================
# 4. LOAD + VERIFY FREEZE MANIFEST
# ==============================================================================

with open(
    FROZEN_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    freeze_manifest = json.load(
        handle
    )


assert isinstance(
    freeze_manifest,
    dict,
), "Freeze manifest is not a JSON object."


assert (
    freeze_manifest.get(
        "status"
    )
    ==
    "FROZEN"
), (
    "Frozen manifest status is not FROZEN."
)


assert (
    freeze_manifest.get(
        "artifact"
    )
    ==
    "cross_encoder_ranked_candidates"
), (
    "Unexpected frozen artifact name."
)


assert (
    freeze_manifest.get(
        "target_used"
    )
    is False
), (
    "Frozen manifest indicates target usage."
)


assert (
    freeze_manifest.get(
        "model"
    )
    ==
    CROSS_ENCODER_MODEL_NAME
), (
    "Frozen model identity mismatch."
)


print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid:",
    "PASS",
)

print(
    "Status:",
    freeze_manifest.get(
        "status"
    ),
)

print(
    "Artifact:",
    freeze_manifest.get(
        "artifact"
    ),
)

print(
    "Model:",
    freeze_manifest.get(
        "model"
    ),
)

print(
    "Target used:",
    freeze_manifest.get(
        "target_used"
    ),
)


# ==============================================================================
# 5. MANIFEST CONTRACT
# ==============================================================================

EXPECTED_FROZEN_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]


assert (
    freeze_manifest.get(
        "columns"
    )
    ==
    EXPECTED_FROZEN_COLUMNS
), (
    "Manifest column contract mismatch."
)


assert (
    int(
        freeze_manifest.get(
            "rows"
        )
    )
    ==
    EXPECTED_R3_ROWS
), (
    "Manifest row contract mismatch."
)


assert (
    int(
        freeze_manifest.get(
            "responses"
        )
    )
    ==
    EXPECTED_R3_RESPONSES
), (
    "Manifest response contract mismatch."
)


assert (
    int(
        freeze_manifest.get(
            "sessions"
        )
    )
    ==
    EXPECTED_R3_SESSIONS
), (
    "Manifest session contract mismatch."
)


assert (
    int(
        freeze_manifest.get(
            "objectives"
        )
    )
    ==
    EXPECTED_R3_OBJECTIVES
), (
    "Manifest objective contract mismatch."
)


print("\n" + "=" * 90)
print("MANIFEST CONTRACT")
print("=" * 90)

print(
    "Rows:",
    f"{int(freeze_manifest['rows']):,}",
)

print(
    "Responses:",
    f"{int(freeze_manifest['responses']):,}",
)

print(
    "Sessions:",
    f"{int(freeze_manifest['sessions']):,}",
)

print(
    "Objectives:",
    f"{int(freeze_manifest['objectives']):,}",
)

print(
    "Columns:",
    "PASS",
)

print(
    "Manifest contract:",
    "PASS",
)


# ==============================================================================
# 6. PARQUET RELOAD
# ==============================================================================

frozen_parquet = pq.ParquetFile(
    FROZEN_CANDIDATE_PATH
)

observed_rows = int(
    frozen_parquet
    .metadata
    .num_rows
)

observed_columns = list(
    frozen_parquet
    .schema_arrow
    .names
)


assert (
    observed_rows
    ==
    EXPECTED_R3_ROWS
), (
    "Frozen parquet row count mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {observed_rows:,}"
)


assert (
    observed_columns
    ==
    EXPECTED_FROZEN_COLUMNS
), (
    "Frozen parquet schema mismatch.\n"
    f"Observed: {observed_columns}"
)


print("\n" + "=" * 90)
print("FROZEN PARQUET RELOAD")
print("=" * 90)

print(
    "Rows:",
    f"{observed_rows:,}",
)

print(
    "Columns:",
    observed_columns,
)

print(
    "Row count:",
    "PASS",
)

print(
    "Schema:",
    "PASS",
)


# ==============================================================================
# 7. SHA256 VERIFICATION
# ==============================================================================

print("\n" + "=" * 90)
print("SHA256 VERIFICATION")
print("=" * 90)


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


observed_sha256 = sha256_file(
    FROZEN_CANDIDATE_PATH
)

expected_sha256 = (
    freeze_manifest.get(
        "candidate_sha256"
    )
)


assert expected_sha256, (
    "Freeze manifest does not contain candidate_sha256."
)


assert (
    observed_sha256
    ==
    expected_sha256
), (
    "Frozen candidate SHA256 mismatch.\n"
    f"Expected: {expected_sha256}\n"
    f"Observed: {observed_sha256}"
)


print(
    "Expected SHA256:",
    expected_sha256,
)

print(
    "Observed SHA256:",
    observed_sha256,
)

print(
    "SHA256:",
    "PASS",
)


# ==============================================================================
# 8. STREAMING FULL DATA AUDIT
# ==============================================================================

print("\n" + "=" * 90)
print("STREAMING FULL DATA AUDIT")
print("=" * 90)

print(
    "Mode:",
    "Parquet scanner / bounded batches",
)

print(
    "No full artifact loaded into pandas.",
)


dataset = ds.dataset(
    FROZEN_CANDIDATE_PATH,
    format="parquet",
)


scanner = dataset.scanner(
    columns=EXPECTED_FROZEN_COLUMNS,
    batch_size=50_000,
)


rows_checked = 0
responses = set()
sessions = set()
objectives = set()

non_finite_scores = 0
invalid_rank_values = 0
negative_rank_values = 0

rank1_rows = 0

# Track exact ordering across physical Parquet batches.
previous_group = None
previous_score = None
previous_turn_index = None
previous_turn_uid = None
previous_rank = None

ordering_violations = 0
rank_sequence_violations = 0

min_rank = None
max_rank = None


# ------------------------------------------------------------------------------
# Per-batch streaming audit
# ------------------------------------------------------------------------------

for record_batch in scanner.to_batches():

    df = (
        record_batch
        .to_pandas()
    )


    batch_rows = len(
        df
    )


    rows_checked += batch_rows


    # --------------------------------------------------------------------------
    # Identity population
    # --------------------------------------------------------------------------

    responses.update(
        df[
            "response_id"
        ]
        .astype(str)
        .tolist()
    )

    sessions.update(
        df[
            "session_id"
        ]
        .astype(str)
        .tolist()
    )

    objectives.update(
        df[
            "objective_uid"
        ]
        .astype(str)
        .tolist()
    )


    # --------------------------------------------------------------------------
    # Score contract
    # --------------------------------------------------------------------------

    score_values = pd.to_numeric(
        df[
            "cross_encoder_score"
        ],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    non_finite_scores += int(
        (~np.isfinite(score_values))
        .sum()
    )


    # --------------------------------------------------------------------------
    # Rank contract
    # --------------------------------------------------------------------------

    rank_values = pd.to_numeric(
        df[
            "cross_encoder_rank"
        ],
        errors="coerce",
    )


    invalid_rank_values += int(
        rank_values.isna().sum()
    )


    rank_array = rank_values.to_numpy(
        dtype=np.float64
    )


    negative_rank_values += int(
        (rank_array < 1).sum()
    )


    finite_rank_mask = np.isfinite(
        rank_array
    )


    if finite_rank_mask.any():

        batch_min = int(
            rank_array[
                finite_rank_mask
            ].min()
        )

        batch_max = int(
            rank_array[
                finite_rank_mask
            ].max()
        )


        if min_rank is None:

            min_rank = batch_min

        else:

            min_rank = min(
                min_rank,
                batch_min,
            )


        if max_rank is None:

            max_rank = batch_max

        else:

            max_rank = max(
                max_rank,
                batch_max,
            )


    rank1_rows += int(
        (
            rank_array
            ==
            1
        ).sum()
    )


    # --------------------------------------------------------------------------
    # Exact score/rank ordering audit
    #
    # Frozen artifact was written ordered by:
    #
    # response_id
    # objective_uid
    # cross_encoder_rank
    #
    # Within each group:
    #
    # score DESC
    # turn_index ASC
    # turn_uid ASC
    # --------------------------------------------------------------------------

    for row in df.itertuples(
        index=False
    ):

        response_id = str(
            row.response_id
        )

        objective_uid = str(
            row.objective_uid
        )

        turn_uid = str(
            row.turn_uid
        )

        turn_index = int(
            row.turn_index
        )

        score = float(
            row.cross_encoder_score
        )

        rank = int(
            row.cross_encoder_rank
        )


        current_group = (
            response_id,
            objective_uid,
        )


        # ----------------------------------------------------------------------
        # Group transition
        # ----------------------------------------------------------------------

        if (
            previous_group
            !=
            current_group
        ):

            # Every new group must begin at rank 1.
            if rank != 1:

                rank_sequence_violations += 1


        else:

            # ------------------------------------------------------------------
            # Rank must increment exactly by one.
            # ------------------------------------------------------------------

            if (
                previous_rank
                is None
                or
                rank
                !=
                previous_rank + 1
            ):

                rank_sequence_violations += 1


            # ------------------------------------------------------------------
            # Score must be non-increasing.
            # ------------------------------------------------------------------

            if (
                previous_score
                is not None
                and
                score
                >
                previous_score
            ):

                ordering_violations += 1


            # ------------------------------------------------------------------
            # If scores tie:
            #
            # turn_index ASC
            # then turn_uid ASC
            # ------------------------------------------------------------------

            if (
                previous_score
                is not None
                and
                score
                ==
                previous_score
            ):

                if (
                    previous_turn_index
                    is not None
                    and
                    turn_index
                    <
                    previous_turn_index
                ):

                    ordering_violations += 1


                elif (
                    previous_turn_index
                    ==
                    turn_index
                    and
                    previous_turn_uid
                    is not None
                    and
                    turn_uid
                    <
                    previous_turn_uid
                ):

                    ordering_violations += 1


        previous_group = current_group
        previous_score = score
        previous_turn_index = turn_index
        previous_turn_uid = turn_uid
        previous_rank = rank


    del record_batch
    del df
    del score_values
    del rank_values
    del rank_array

    gc.collect()


# ==============================================================================
# 9. FULL DATA ASSERTIONS
# ==============================================================================

assert (
    rows_checked
    ==
    EXPECTED_R3_ROWS
), (
    "Full frozen row audit mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {rows_checked:,}"
)


assert (
    len(responses)
    ==
    EXPECTED_R3_RESPONSES
), (
    "Response population mismatch."
)


assert (
    len(sessions)
    ==
    EXPECTED_R3_SESSIONS
), (
    "Session population mismatch."
)


assert (
    len(objectives)
    ==
    EXPECTED_R3_OBJECTIVES
), (
    "Objective population mismatch."
)


assert (
    non_finite_scores
    ==
    0
), (
    "Non-finite Cross-Encoder scores detected."
)


assert (
    invalid_rank_values
    ==
    0
), (
    "Invalid/non-numeric rank values detected."
)


assert (
    negative_rank_values
    ==
    0
), (
    "Rank values below 1 detected."
)


assert (
    min_rank
    ==
    1
), (
    f"Unexpected minimum rank: {min_rank}"
)


assert (
    ordering_violations
    ==
    0
), (
    "Cross-Encoder score/tie-break ordering violations detected.\n"
    f"Violations: {ordering_violations}"
)


assert (
    rank_sequence_violations
    ==
    0
), (
    "Rank sequence violations detected.\n"
    f"Violations: {rank_sequence_violations}"
)


print(
    "Rows checked:",
    f"{rows_checked:,}",
)

print(
    "Responses:",
    f"{len(responses):,}",
)

print(
    "Sessions:",
    f"{len(sessions):,}",
)

print(
    "Objectives:",
    f"{len(objectives):,}",
)

print(
    "Non-finite scores:",
    non_finite_scores,
)

print(
    "Invalid ranks:",
    invalid_rank_values,
)

print(
    "Minimum rank:",
    min_rank,
)

print(
    "Maximum rank:",
    max_rank,
)

print(
    "Rank-1 rows:",
    f"{rank1_rows:,}",
)

print(
    "Ordering violations:",
    ordering_violations,
)

print(
    "Rank sequence violations:",
    rank_sequence_violations,
)

print(
    "Full streaming audit:",
    "PASS",
)


# ==============================================================================
# 10. TARGET ISOLATION
# ==============================================================================

FORBIDDEN_COLUMNS = {
    "target",
    "label",
    "is_correct",
    "mastery_label",
}


observed_forbidden = (
    set(
        observed_columns
    )
    &
    FORBIDDEN_COLUMNS
)


assert not observed_forbidden, (
    "Target-bearing columns found:\n"
    f"{sorted(observed_forbidden)}"
)


print("\n" + "=" * 90)
print("TARGET ISOLATION")
print("=" * 90)

print(
    "Forbidden columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 11. FREEZE MANIFEST RECONFIRMATION
# ==============================================================================

assert (
    freeze_manifest[
        "candidate_sha256"
    ]
    ==
    observed_sha256
)

assert (
    freeze_manifest[
        "rows"
    ]
    ==
    observed_rows
)

assert (
    freeze_manifest[
        "columns"
    ]
    ==
    observed_columns
)


print("\n" + "=" * 90)
print("FREEZE MANIFEST RECONFIRMATION")
print("=" * 90)

print(
    "Manifest SHA256:",
    "PASS",
)

print(
    "Manifest row count:",
    "PASS",
)

print(
    "Manifest schema:",
    "PASS",
)

print(
    "Frozen contract:",
    "PASS",
)


# ==============================================================================
# 12. CELL GATE
# ==============================================================================

CROSS_ENCODER_CELL_7_READY = True
CROSS_ENCODER_FROZEN_RELOAD_VERIFIED = True


print("\n" + "=" * 90)
print("CELL 7 FINAL STATUS")
print("=" * 90)

print(
    "Frozen artifact:",
    "PASS",
)

print(
    "Manifest:",
    "PASS",
)

print(
    "SHA256:",
    "PASS",
)

print(
    "Population:",
    "PASS",
)

print(
    "Schema:",
    "PASS",
)

print(
    "Score finiteness:",
    "PASS",
)

print(
    "Rank sequence:",
    "PASS",
)

print(
    "Score ordering:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "CROSS_ENCODER_FROZEN_RELOAD_VERIFIED:",
    CROSS_ENCODER_FROZEN_RELOAD_VERIFIED,
)

print(
    "CROSS_ENCODER_CELL_7_READY:",
    CROSS_ENCODER_CELL_7_READY,
)


assert (
    CROSS_ENCODER_FROZEN_RELOAD_VERIFIED
    is True
)

assert (
    CROSS_ENCODER_CELL_7_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 7 — FROZEN RELOAD VERIFICATION: PASS"
)
print("=" * 90)


# ==============================================================================
# 13. MEMORY CLEANUP
# ==============================================================================

cleanup_names = [
    "frozen_parquet",
    "dataset",
    "scanner",
    "responses",
    "sessions",
    "objectives",
    "freeze_manifest",
    "rank_array",
    "score_values",
    "rank_values",
]


for name in cleanup_names:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


print(
    "Cell 7 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 7 — FROZEN RELOAD + INTEGRITY VERIFICATION

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS
Cell 4 dependency : PASS
Cell 5 dependency : PASS
Cell 6 dependency : PASS

FROZEN ARTIFACT PATHS
Frozen root: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen
Candidate: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cross_encoder_ranked_candidates.parquet
Manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cell6_freeze_manifest.json

ARTIFACT EXISTENCE
Frozen directory: True
Candidate parquet: True
Freeze manifest: True

FREEZE MANIFEST
JSON valid: PASS
Status: FROZEN
Artifact: cross_encoder_ranked_candidates
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Target used: False

MANIFEST CONTRACT
Rows: 2,482,137
Responses: 35,072
Sessions: 22,821
Ob

In [14]:
# ==============================================================================
# TRACE THE ACE — CROSS-ENCODER RERANKING
# CELL 8 — FINAL AUDIT / FREEZE GATE
# ==============================================================================

from __future__ import annotations

import gc
import hashlib
import json
from pathlib import Path

import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — CROSS-ENCODER RERANKING")
print("CELL 8 — FINAL AUDIT / FREEZE GATE")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "CROSS_ENCODER_CELL_0_READY" in globals()
    and CROSS_ENCODER_CELL_0_READY is True
), "Cell 0 is not ready."

assert (
    "CROSS_ENCODER_CELL_1_READY" in globals()
    and CROSS_ENCODER_CELL_1_READY is True
), "Cell 1 is not ready."

assert (
    "CROSS_ENCODER_CELL_2_READY" in globals()
    and CROSS_ENCODER_CELL_2_READY is True
), "Cell 2 is not ready."

assert (
    "CROSS_ENCODER_CELL_3_READY" in globals()
    and CROSS_ENCODER_CELL_3_READY is True
), "Cell 3 is not ready."

assert (
    "CROSS_ENCODER_CELL_4_READY" in globals()
    and CROSS_ENCODER_CELL_4_READY is True
), "Cell 4 is not ready."

assert (
    "CROSS_ENCODER_CELL_5_READY" in globals()
    and CROSS_ENCODER_CELL_5_READY is True
), "Cell 5 is not ready."

assert (
    "CROSS_ENCODER_CELL_6_READY" in globals()
    and CROSS_ENCODER_CELL_6_READY is True
), "Cell 6 is not ready."

assert (
    "CROSS_ENCODER_CELL_7_READY" in globals()
    and CROSS_ENCODER_CELL_7_READY is True
), "Cell 7 is not ready."


print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

for i in range(8):
    print(
        f"Cell {i} dependency : PASS"
    )


# ==============================================================================
# 2. FROZEN PATHS
# ==============================================================================

FREEZE_ROOT = (
    CROSS_ENCODER_ROOT
    / "frozen"
)

FROZEN_CANDIDATE_PATH = (
    FREEZE_ROOT
    / "cross_encoder_ranked_candidates.parquet"
)

FROZEN_MANIFEST_PATH = (
    FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)


print("\n" + "=" * 90)
print("FROZEN ARTIFACT PATHS")
print("=" * 90)

print(
    "Frozen root :",
    FREEZE_ROOT,
)

print(
    "Candidates  :",
    FROZEN_CANDIDATE_PATH,
)

print(
    "Manifest    :",
    FROZEN_MANIFEST_PATH,
)


# ==============================================================================
# 3. ARTIFACT EXISTENCE
# ==============================================================================

assert FREEZE_ROOT.exists(), (
    "Frozen Cross-Encoder root is missing."
)

assert FROZEN_CANDIDATE_PATH.exists(), (
    "Frozen Cross-Encoder candidate artifact is missing."
)

assert FROZEN_MANIFEST_PATH.exists(), (
    "Frozen Cross-Encoder manifest is missing."
)


print("\n" + "=" * 90)
print("ARTIFACT EXISTENCE")
print("=" * 90)

print(
    "Frozen root:",
    True,
)

print(
    "Candidate parquet:",
    True,
)

print(
    "Freeze manifest:",
    True,
)


# ==============================================================================
# 4. LOAD MANIFEST
# ==============================================================================

with open(
    FROZEN_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as handle:

    manifest = json.load(
        handle
    )


assert isinstance(
    manifest,
    dict,
), "Freeze manifest is not a JSON object."


print("\n" + "=" * 90)
print("FREEZE MANIFEST")
print("=" * 90)

print(
    "JSON valid:",
    True,
)

print(
    "Status:",
    manifest.get("status"),
)

print(
    "Artifact:",
    manifest.get("artifact"),
)

print(
    "Model:",
    manifest.get("model"),
)

print(
    "Target used:",
    manifest.get("target_used"),
)


assert (
    manifest.get("status")
    ==
    "FROZEN"
), "Final freeze status is not FROZEN."


assert (
    manifest.get("artifact")
    ==
    "cross_encoder_ranked_candidates"
), "Unexpected frozen artifact."


assert (
    manifest.get("model")
    ==
    CROSS_ENCODER_MODEL_NAME
), "Cross-Encoder model mismatch."


assert (
    manifest.get("target_used")
    is False
), "Target usage detected in final manifest."


# ==============================================================================
# 5. FINAL SCHEMA CONTRACT
# ==============================================================================

EXPECTED_FINAL_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "turn_uid",
    "role",
    "turn_index",
    "cross_encoder_score",
    "cross_encoder_rank",
]


assert (
    manifest.get("columns")
    ==
    EXPECTED_FINAL_COLUMNS
), (
    "Final manifest schema contract mismatch."
)


print("\n" + "=" * 90)
print("FINAL SCHEMA CONTRACT")
print("=" * 90)

print(
    "Columns:",
    manifest.get("columns"),
)

print(
    "Schema contract:",
    "PASS",
)


# ==============================================================================
# 6. FROZEN PARQUET CONTRACT
# ==============================================================================

frozen_parquet = pq.ParquetFile(
    FROZEN_CANDIDATE_PATH
)

observed_rows = int(
    frozen_parquet
    .metadata
    .num_rows
)

observed_columns = list(
    frozen_parquet
    .schema_arrow
    .names
)


assert (
    observed_rows
    ==
    EXPECTED_R3_ROWS
), (
    "Final frozen population mismatch.\n"
    f"Expected: {EXPECTED_R3_ROWS:,}\n"
    f"Observed: {observed_rows:,}"
)


assert (
    observed_columns
    ==
    EXPECTED_FINAL_COLUMNS
), (
    "Final frozen schema mismatch."
)


assert (
    int(
        manifest.get("rows")
    )
    ==
    observed_rows
), (
    "Manifest/parquet row count mismatch."
)


print("\n" + "=" * 90)
print("FINAL FROZEN PARQUET CONTRACT")
print("=" * 90)

print(
    "Rows:",
    f"{observed_rows:,}",
)

print(
    "Responses:",
    f"{int(manifest.get('responses')):,}",
)

print(
    "Sessions:",
    f"{int(manifest.get('sessions')):,}",
)

print(
    "Objectives:",
    f"{int(manifest.get('objectives')):,}",
)

print(
    "Schema:",
    "PASS",
)

print(
    "Population:",
    "PASS",
)


# ==============================================================================
# 7. SHA256 RECHECK
# ==============================================================================

def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


observed_sha256 = sha256_file(
    FROZEN_CANDIDATE_PATH
)

manifest_sha256 = (
    manifest.get(
        "candidate_sha256"
    )
)


assert manifest_sha256, (
    "Manifest does not contain candidate_sha256."
)

assert (
    observed_sha256
    ==
    manifest_sha256
), (
    "Final frozen SHA256 mismatch."
)


print("\n" + "=" * 90)
print("FINAL SHA256 AUDIT")
print("=" * 90)

print(
    "Manifest SHA256:",
    manifest_sha256,
)

print(
    "Observed SHA256:",
    observed_sha256,
)

print(
    "SHA256:",
    "PASS",
)


# ==============================================================================
# 8. FINAL TARGET ISOLATION
# ==============================================================================

FORBIDDEN_TARGET_COLUMNS = {
    "target",
    "label",
    "is_correct",
    "mastery_label",
}


target_columns_found = (
    set(observed_columns)
    &
    FORBIDDEN_TARGET_COLUMNS
)


assert not target_columns_found, (
    "Target-bearing columns found in final frozen artifact:\n"
    f"{sorted(target_columns_found)}"
)


print("\n" + "=" * 90)
print("FINAL TARGET ISOLATION")
print("=" * 90)

print(
    "Forbidden target columns:",
    [],
)

print(
    "Target leakage:",
    "NONE",
)

print(
    "Target isolation:",
    "PASS",
)


# ==============================================================================
# 9. UPSTREAM CONTRACT CHAIN
# ==============================================================================

assert (
    CROSS_ENCODER_CELL_6_READY
    is True
)

assert (
    CROSS_ENCODER_CELL_7_READY
    is True
)

assert (
    CROSS_ENCODER_FROZEN_RELOAD_VERIFIED
    is True
)


print("\n" + "=" * 90)
print("UPSTREAM CONTRACT CHAIN")
print("=" * 90)

print(
    "Cell 6 final freeze:",
    "PASS",
)

print(
    "Cell 7 frozen reload:",
    "PASS",
)

print(
    "Cell 7 integrity verification:",
    "PASS",
)

print(
    "Upstream chain:",
    "PASS",
)


# ==============================================================================
# 10. FINAL MANIFEST CONSISTENCY
# ==============================================================================

assert (
    manifest.get("ranking_group")
    ==
    [
        "response_id",
        "objective_uid",
    ]
), "Ranking group contract mismatch."


assert (
    manifest.get("primary_order")
    ==
    "cross_encoder_score DESC"
), "Primary ranking contract mismatch."


assert (
    manifest.get("secondary_order")
    ==
    "turn_index ASC"
), "Secondary ranking contract mismatch."


assert (
    manifest.get("tertiary_order")
    ==
    "turn_uid ASC"
), "Tertiary ranking contract mismatch."


assert (
    manifest.get("score_transformation")
    ==
    "NONE"
), "Score transformation contract mismatch."


print("\n" + "=" * 90)
print("RANKING CONTRACT")
print("=" * 90)

print(
    "Ranking group:",
    manifest.get("ranking_group"),
)

print(
    "Primary order:",
    manifest.get("primary_order"),
)

print(
    "Secondary order:",
    manifest.get("secondary_order"),
)

print(
    "Tertiary order:",
    manifest.get("tertiary_order"),
)

print(
    "Score transformation:",
    manifest.get("score_transformation"),
)

print(
    "Ranking contract:",
    "PASS",
)


# ==============================================================================
# 11. FINAL FREEZE GATE
# ==============================================================================

CROSS_ENCODER_FINAL_FREEZE = True
CROSS_ENCODER_COMPLETE = True
CROSS_ENCODER_CELL_8_READY = True


print("\n" + "=" * 90)
print("FINAL FREEZE GATE")
print("=" * 90)

print(
    "Frozen artifact:",
    True,
)

print(
    "Frozen manifest:",
    True,
)

print(
    "Population:",
    "PASS",
)

print(
    "Schema:",
    "PASS",
)

print(
    "SHA256:",
    "PASS",
)

print(
    "Target isolation:",
    "PASS",
)

print(
    "Ranking contract:",
    "PASS",
)

print(
    "Cell 6 freeze:",
    "PASS",
)

print(
    "Cell 7 verification:",
    "PASS",
)

print(
    "CROSS_ENCODER_FINAL_FREEZE:",
    CROSS_ENCODER_FINAL_FREEZE,
)

print(
    "CROSS_ENCODER_COMPLETE:",
    CROSS_ENCODER_COMPLETE,
)

print(
    "CROSS_ENCODER_CELL_8_READY:",
    CROSS_ENCODER_CELL_8_READY,
)


assert (
    CROSS_ENCODER_FINAL_FREEZE
    is True
)

assert (
    CROSS_ENCODER_COMPLETE
    is True
)

assert (
    CROSS_ENCODER_CELL_8_READY
    is True
)


print("\n" + "=" * 90)
print(
    "08 CROSS-ENCODER CELL 8 — FINAL AUDIT / FREEZE GATE: PASS"
)
print("=" * 90)


# ==============================================================================
# 12. MEMORY CLEANUP
# ==============================================================================

cleanup_names = [
    "frozen_parquet",
    "manifest",
]

for name in cleanup_names:

    if name in globals():

        try:
            del globals()[name]
        except Exception:
            pass


gc.collect()


print(
    "Cell 8 memory cleanup: PASS"
)

TRACE THE ACE — CROSS-ENCODER RERANKING
CELL 8 — FINAL AUDIT / FREEZE GATE

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS
Cell 2 dependency : PASS
Cell 3 dependency : PASS
Cell 4 dependency : PASS
Cell 5 dependency : PASS
Cell 6 dependency : PASS
Cell 7 dependency : PASS

FROZEN ARTIFACT PATHS
Frozen root : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen
Candidates  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cross_encoder_ranked_candidates.parquet
Manifest    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\cross_encoder\frozen\cell6_freeze_manifest.json

ARTIFACT EXISTENCE
Frozen root: True
Candidate parquet: True
Freeze manifest: True

FREEZE MANIFEST
JSON valid: True
Status: FROZEN
Artifact: cross_encoder_ranked_candidates
Model: cross-encoder/ms-marco-MiniLM-L6-v2
Target used: False

FINAL SCHEMA CONTRACT
Columns: ['response_id', 'session_